In [0]:
bronze_base = "abfss://autorent@stautorentjs2026.dfs.core.windows.net/bronze"
silver_base = "abfss://autorent@stautorentjs2026.dfs.core.windows.net/silver"

cars_source = f"{bronze_base}/day1/Cars.txt"
cars_silver = f"{silver_base}/cars"

print("Cars source:", cars_source)
print("Cars silver:", cars_silver)


In [0]:
cars_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(cars_source)
)

display(cars_df)

In [0]:
cars_df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim

cars_clean_df = (
    cars_df
    .select(
        trim(col("PlateNo")).alias("PlateNo"),
        trim(col("Car Name")).alias("CarName"),
        trim(col("Manufacturer")).alias("Manufacturer"),
        col("Year Model").cast("int").alias("YearModel"),
        trim(col("Car Body Type")).alias("CarBodyType"),
        trim(col("Transmission")).alias("Transmission"),
        col("Daily Rate (in Peso)")
            .cast("decimal(10,2)")
            .alias("DailyRate")
    )
)

display(cars_clean_df)

In [0]:
cars_clean_df.printSchema()

In [0]:
print("Row count:", cars_clean_df.count())

cars_clean_df.select(
    "PlateNo",
    "CarName",
    "DailyRate"
).show(truncate=False)


In [0]:
(
    cars_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(cars_silver)
)

print("Day 1 Cars successfully written to Silver.")

In [0]:
customers_source = f"{bronze_base}/day1/Customers.txt"
customers_silver = f"{silver_base}/customers"

customers_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(customers_source)
)

display(customers_df)

In [0]:
print(customers_df.columns)
customers_df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim

customers_clean_df = (
    customers_df
    .select(
        col("`CustomerNo`").cast("int").alias("CustomerNo"),
        trim(col("`Name`")).alias("CustomerName"),
        trim(col("`Address`")).alias("Address"),
        trim(col("`Driver's License No.`")).alias("DriversLicenseNo"),
        col("`Age`").cast("int").alias("Age"),
        trim(col("`Email`")).alias("Email"),
        trim(col("`Mobile No.`")).alias("MobileNo")
    )
)

display(customers_clean_df)

In [0]:
customers_clean_df.printSchema()

print("Row count:", customers_clean_df.count())

In [0]:
(
    customers_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(customers_silver)
)

print("Day 1 Customers successfully written to Silver.")

In [0]:
employees_source = f"{bronze_base}/day1/Employees.txt"
employees_silver = f"{silver_base}/employees"

employees_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(employees_source)
)

display(employees_df)

In [0]:
print(employees_df.columns)
employees_df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, to_date

employees_clean_df = (
    employees_df
    .select(
        col("`EmployeeNo`").cast("int").alias("EmployeeNo"),
        trim(col("`Name`")).alias("EmployeeName"),
        trim(col("`Address`")).alias("Address"),
        to_date(col("`Birthdate`")).alias("BirthDate"),
        trim(col("`Email`")).alias("Email"),
        trim(col("`Phone`")).alias("Phone")
    )
)

display(employees_clean_df)

In [0]:
employees_clean_df.printSchema()

print("Row count:", employees_clean_df.count())

employees_clean_df.show(truncate=False)

In [0]:
(
    employees_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(employees_silver)
)

print("Day 1 Employees successfully written to Silver.")

In [0]:
locations_source = f"{bronze_base}/day1/Location.txt"
locations_silver = f"{silver_base}/locations"

locations_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(locations_source)
)

display(locations_df)

In [0]:
print(locations_df.columns)
locations_df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim

locations_clean_df = (
    locations_df
    .select(
        trim(col("`LocationCode`")).alias("LocationCode"),
        trim(col("`Address1`")).alias("Address1"),
        trim(col("`Address2`")).alias("Address2"),
        trim(col("`City`")).alias("City"),
        trim(col("`Province`")).alias("Province")
    )
)

display(locations_clean_df)

In [0]:
locations_clean_df.printSchema()

print("Row count:", locations_clean_df.count())

locations_clean_df.show(truncate=False)

In [0]:
(
    locations_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(locations_silver)
)

print("Day 1 Locations successfully written to Silver.")

In [0]:
transactions_source = f"{bronze_base}/day1/Transactions.txt"
transactions_silver = f"{silver_base}/transactions"

transactions_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(transactions_source)
)

display(transactions_df)

In [0]:
print(transactions_df.columns)
transactions_df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, to_timestamp

transactions_clean_df = (
    transactions_df
    .select(
        trim(col("`plateno`")).alias("PlateNo"),
        trim(col("`locationcode`")).alias("LocationCode"),
        col("`customerno`").cast("int").alias("CustomerNo"),
        col("`employeeno`").cast("int").alias("EmployeeNo"),
        to_timestamp(
            col("`date`"),
            "yyyy-MM-dd HH:mm:ss.SSS"
        ).alias("TransactionDate"),
        col("`qty`").cast("int").alias("Quantity"),
        col("`amount`").cast("decimal(12,2)").alias("Amount"),
        trim(col("`type`")).alias("TransactionType"),
        col("`days`").cast("int").alias("Days")
    )
)

display(transactions_clean_df)

In [0]:
transactions_clean_df.printSchema()

print(
    "Day 1 transaction row count:",
    transactions_clean_df.count()
)

transactions_clean_df.show(20, truncate=False)

In [0]:
transactions_clean_df.groupBy(
    "TransactionType"
).count().show()

In [0]:
(
    transactions_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(transactions_silver)
)

print("Day 1 Transactions successfully written to Silver.")


In [0]:
day2_base = f"{bronze_base}/day2"
day2_silver_base = f"{silver_base}/day2"

print("Day 2 Bronze:", day2_base)
print("Day 2 Silver:", day2_silver_base)

In [0]:
day2_cars_source = f"{day2_base}/Cars.txt"
day2_cars_silver = f"{day2_silver_base}/cars"

day2_cars_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day2_cars_source)
)

display(day2_cars_df)

In [0]:
from pyspark.sql.functions import col, trim

day2_cars_clean_df = (
    day2_cars_df
    .select(
        trim(col("`PlateNo`")).alias("PlateNo"),
        trim(col("`Car Name`")).alias("CarName"),
        trim(col("`Manufacturer`")).alias("Manufacturer"),
        col("`Year Model`").cast("int").alias("YearModel"),
        trim(col("`Car Body Type`")).alias("CarBodyType"),
        trim(col("`Transmission`")).alias("Transmission"),
        col("`Daily Rate (in Peso)`")
            .cast("decimal(10,2)")
            .alias("DailyRate")
    )
)

display(day2_cars_clean_df)

In [0]:
day2_cars_clean_df.printSchema()

print("Day 2 Cars row count:", day2_cars_clean_df.count())

In [0]:
(
    day2_cars_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day2_cars_silver)
)

print("Day 2 Cars successfully written to Silver.")

In [0]:
day2_customers_source = f"{day2_base}/Customers.txt"
day2_customers_silver = f"{day2_silver_base}/customers"

day2_customers_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day2_customers_source)
)

display(day2_customers_df)

In [0]:
from pyspark.sql.functions import col, trim

day2_customers_clean_df = (
    day2_customers_df
    .select(
        col("`CustomerNo`").cast("int").alias("CustomerNo"),
        trim(col("`Name`")).alias("CustomerName"),
        trim(col("`Address`")).alias("Address"),
        trim(col("`Driver's License No.`")).alias("DriversLicenseNo"),
        col("`Age`").cast("int").alias("Age"),
        trim(col("`Email`")).alias("Email"),
        trim(col("`Mobile No.`")).alias("MobileNo")
    )
)

display(day2_customers_clean_df)

In [0]:
day2_customers_clean_df.printSchema()

print(
    "Day 2 Customers row count:",
    day2_customers_clean_df.count()
)

In [0]:
(
    day2_customers_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day2_customers_silver)
)

print("Day 2 Customers successfully written to Silver.")

In [0]:
day2_employees_source = f"{day2_base}/Employees.txt"
day2_employees_silver = f"{day2_silver_base}/employees"

day2_employees_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day2_employees_source)
)

display(day2_employees_df)

In [0]:
from pyspark.sql.functions import col, trim, to_date

day2_employees_clean_df = (
    day2_employees_df
    .select(
        col("`EmployeeNo`").cast("int").alias("EmployeeNo"),
        trim(col("`Name`")).alias("EmployeeName"),
        trim(col("`Address`")).alias("Address"),
        to_date(col("`Birthdate`")).alias("BirthDate"),
        trim(col("`Email`")).alias("Email"),
        trim(col("`Phone`")).alias("Phone")
    )
)

display(day2_employees_clean_df)

In [0]:
day2_employees_clean_df.printSchema()

print(
    "Day 2 Employees row count:",
    day2_employees_clean_df.count()
)

day2_employees_clean_df.show(truncate=False)

In [0]:
(
    day2_employees_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day2_employees_silver)
)

print("Day 2 Employees successfully written to Silver.")

In [0]:
day2_locations_source = f"{day2_base}/Location.txt"
day2_locations_silver = f"{day2_silver_base}/locations"

day2_locations_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day2_locations_source)
)

display(day2_locations_df)

In [0]:
from pyspark.sql.functions import col, trim

day2_locations_clean_df = (
    day2_locations_df
    .select(
        trim(col("`LocationCode`")).alias("LocationCode"),
        trim(col("`Address1`")).alias("Address1"),
        trim(col("`Address2`")).alias("Address2"),
        trim(col("`City`")).alias("City"),
        trim(col("`Province`")).alias("Province")
    )
)

display(day2_locations_clean_df)

In [0]:
day2_locations_clean_df.printSchema()

print(
    "Day 2 Locations row count:",
    day2_locations_clean_df.count()
)

day2_locations_clean_df.show(truncate=False)

In [0]:
(
    day2_locations_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day2_locations_silver)
)

print("Day 2 Locations successfully written to Silver.")

In [0]:
day2_transactions_source = f"{day2_base}/Transactions.txt"
day2_transactions_silver = f"{day2_silver_base}/transactions"

day2_transactions_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day2_transactions_source)
)

display(day2_transactions_df)

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    coalesce,
    try_to_timestamp,
    lit
)

day2_transactions_clean_df = (
    day2_transactions_df
    .select(
        trim(col("`plateno`")).alias("PlateNo"),
        trim(col("`locationcode`")).alias("LocationCode"),
        col("`customerno`").cast("int").alias("CustomerNo"),
        col("`employeeno`").cast("int").alias("EmployeeNo"),

        coalesce(
            try_to_timestamp(
                col("`date`"),
                lit("M/d/yyyy H:mm")
            ),
            try_to_timestamp(
                col("`date`"),
                lit("M/d/yyyy H:mm:ss")
            ),
            try_to_timestamp(
                col("`date`"),
                lit("yyyy-MM-dd HH:mm:ss.SSS")
            )
        ).alias("TransactionDate"),

        col("`qty`").cast("int").alias("Quantity"),
        col("`amount`")
            .cast("decimal(12,2)")
            .alias("Amount"),
        trim(col("`type`")).alias("TransactionType"),
        col("`days`").cast("int").alias("Days")
    )
)

display(day2_transactions_clean_df)

In [0]:
day2_transactions_clean_df.printSchema()

print(
    "Day 2 Transactions row count:",
    day2_transactions_clean_df.count()
)

day2_transactions_clean_df.groupBy(
    "TransactionType"
).count().show()

In [0]:
(
    day2_transactions_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day2_transactions_silver)
)

print("Day 2 Transactions successfully written to Silver.")

In [0]:
day2_transactions_clean_df.printSchema()

print(
    "Day 2 Transactions row count:",
    day2_transactions_clean_df.count()
)

display(
    day2_transactions_clean_df.select(
        "PlateNo",
        "TransactionDate",
        "Amount",
        "TransactionType",
        "Days"
    )
)

In [0]:
day1_cars = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/cars")
)

day2_cars = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/cars")
)

print("Day 1 Cars:", day1_cars.count())
print("Day 2 Cars:", day2_cars.count())

display(day1_cars)
display(day2_cars)

In [0]:
print("Day 1 duplicate PlateNo values:")

day1_cars.groupBy("PlateNo") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

print("Day 2 duplicate PlateNo values:")

day2_cars.groupBy("PlateNo") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

In [0]:
from pyspark.sql.functions import (
    col,
    sha2,
    concat_ws,
    coalesce,
    lit
)

car_attributes = [
    "CarName",
    "Manufacturer",
    "YearModel",
    "CarBodyType",
    "Transmission",
    "DailyRate"
]

def add_car_hash(dataframe):
    return dataframe.withColumn(
        "AttributeHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in car_attributes
                ]
            ),
            256
        )
    )

day1_cars_hashed = add_car_hash(day1_cars)

day2_cars_hashed = add_car_hash(day2_cars)

In [0]:
from pyspark.sql.functions import when

cars_comparison_df = (
    day2_cars_hashed.alias("day2")
    .join(
        day1_cars_hashed.alias("day1"),
        col("day2.PlateNo") == col("day1.PlateNo"),
        "left"
    )
    .select(
        col("day2.PlateNo").alias("PlateNo"),
        col("day2.CarName").alias("Day2CarName"),
        col("day1.DailyRate").alias("Day1DailyRate"),
        col("day2.DailyRate").alias("Day2DailyRate"),

        when(
            col("day1.PlateNo").isNull(),
            lit("NEW")
        )
        .when(
            col("day1.AttributeHash")
            != col("day2.AttributeHash"),
            lit("CHANGED")
        )
        .otherwise(
            lit("UNCHANGED")
        )
        .alias("ChangeType")
    )
)

display(
    cars_comparison_df.orderBy(
        "ChangeType",
        "PlateNo"
    )
)

In [0]:
cars_comparison_df.groupBy(
    "ChangeType"
).count().orderBy(
    "ChangeType"
).show()

In [0]:
display(
    cars_comparison_df
    .filter(
        col("ChangeType").isin(
            "NEW",
            "CHANGED"
        )
    )
    .orderBy("PlateNo")
)

In [0]:
gold_base = (
    "abfss://autorent@stautorentjs2026"
    ".dfs.core.windows.net/gold"
)

dim_car_path = f"{gold_base}/dim_car"

print("Gold base:", gold_base)
print("dim_car path:", dim_car_path)

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    sha2,
    concat_ws,
    coalesce
)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

car_attributes = [
    "CarName",
    "Manufacturer",
    "YearModel",
    "CarBodyType",
    "Transmission",
    "DailyRate"
]

day1_car_dimension_df = (
    day1_cars
    .withColumn(
        "AttributeHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in car_attributes
                ]
            ),
            256
        )
    )
    .withColumn(
        "CarKey",
        row_number().over(
            Window.orderBy("PlateNo")
        )
    )
    .withColumn(
        "EffectiveDate",
        current_timestamp()
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "CarKey",
        "PlateNo",
        "CarName",
        "Manufacturer",
        "YearModel",
        "CarBodyType",
        "Transmission",
        "DailyRate",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(day1_car_dimension_df)

In [0]:
day1_car_dimension_df.printSchema()

print(
    "Initial dim_car row count:",
    day1_car_dimension_df.count()
)

day1_car_dimension_df.groupBy(
    "IsCurrent"
).count().show()

In [0]:
day1_car_dimension_df.groupBy(
    "PlateNo"
).count().filter(
    col("count") > 1
).show()

In [0]:
from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark, dim_car_path):
    (
        day1_car_dimension_df.write
        .format("delta")
        .mode("overwrite")
        .save(dim_car_path)
    )

    print("Initial dim_car created.")
else:
    print("dim_car already exists. Initialization skipped.")

In [0]:
gold_dim_car_df = (
    spark.read
    .format("delta")
    .load(dim_car_path)
)

display(
    gold_dim_car_df.orderBy(
        "PlateNo",
        "EffectiveDate"
    )
)

In [0]:
day2_car_stage_df = (
    day2_cars
    .withColumn(
        "AttributeHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in car_attributes
                ]
            ),
            256
        )
    )
)

display(
    day2_car_stage_df.orderBy("PlateNo")
)

In [0]:
from pyspark.sql.functions import col

current_gold_cars_df = (
    spark.read
    .format("delta")
    .load(dim_car_path)
    .filter(col("IsCurrent") == True)
)

day2_car_changes_df = (
    day2_car_stage_df.alias("source")
    .join(
        current_gold_cars_df.alias("target"),
        col("source.PlateNo") == col("target.PlateNo"),
        "left"
    )
    .filter(
        col("target.PlateNo").isNull()
        |
        (
            col("source.AttributeHash")
            != col("target.AttributeHash")
        )
    )
    .select("source.*")
)

display(
    day2_car_changes_df.orderBy("PlateNo")
)

print(
   "Day 2 new or changed Cars:",
   day2_car_changes_df.count()
)

In [0]:
day2_processing_timestamp = (
    spark.sql(
        "SELECT current_timestamp() AS ProcessingTimestamp"
    )
    .first()["ProcessingTimestamp"]
)

print(
    "Day 2 processing timestamp:",
    day2_processing_timestamp
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import lit

dim_car_delta = DeltaTable.forPath(
    spark,
    dim_car_path
)

(
    dim_car_delta.alias("target")
    .merge(
        day2_car_stage_df.alias("source"),
        """
        target.PlateNo = source.PlateNo
        AND target.IsCurrent = true
        """
    )
    .whenMatchedUpdate(
        condition="""
        target.AttributeHash <> source.AttributeHash
        """,
        set={
            "ExpirationDate": lit(day2_processing_timestamp),
            "IsCurrent": lit(False)
        }
    )
    .execute()
)

print("Changed Day 1 Car versions were expired.")

In [0]:
gold_after_expiry_df = (
    spark.read
    .format("delta")
    .load(dim_car_path)
)

display(
    gold_after_expiry_df
    .filter(col("IsCurrent") == False)
    .orderBy("PlateNo")
)

In [0]:
current_max_car_key = (
    gold_after_expiry_df
    .selectExpr(
        "COALESCE(MAX(CarKey), 0) AS MaxCarKey"
    )
    .first()["MaxCarKey"]
)

print(
    "Current maximum CarKey:",
    current_max_car_key
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    lit,
    row_number
)

day2_new_versions_df = (
    day2_car_changes_df
    .withColumn(
        "CarKey",
        row_number().over(
            Window.orderBy("PlateNo")
        ) + lit(current_max_car_key)
    )
    .withColumn(
        "EffectiveDate",
        lit(day2_processing_timestamp).cast("timestamp")
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "CarKey",
        "PlateNo",
        "CarName",
        "Manufacturer",
        "YearModel",
        "CarBodyType",
        "Transmission",
        "DailyRate",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(
    day2_new_versions_df.orderBy("PlateNo")
)

In [0]:
day2_new_version_count = day2_new_versions_df.count()

if day2_new_version_count > 0:
    (
        day2_new_versions_df.write
        .format("delta")
        .mode("append")
        .save(dim_car_path)
    )

    print(
        "Day 2 Car versions inserted:",
        day2_new_version_count
    )
else:
    print(
        "No new or changed Day 2 Cars were found."
    )

In [0]:
final_dim_car_df = (
    spark.read
    .format("delta")
    .load(dim_car_path)
)

display(
    final_dim_car_df.orderBy(
        "PlateNo",
        "EffectiveDate"
    )
)


In [0]:
final_dim_car_df.groupBy(
    "IsCurrent"
).count().show()

In [0]:
cars_with_history_df = (
    final_dim_car_df
    .groupBy("PlateNo")
    .count()
    .filter(col("count") > 1)
)

display(
    cars_with_history_df.orderBy("PlateNo")
)

In [0]:
display(
    final_dim_car_df.alias("dimension")
    .join(
        cars_with_history_df
        .select("PlateNo")
        .alias("history"),
        col("dimension.PlateNo")
        == col("history.PlateNo"),
        "inner"
    )
    .select(
        col("dimension.CarKey"),
        col("dimension.PlateNo"),
        col("dimension.CarName"),
        col("dimension.Manufacturer"),
        col("dimension.DailyRate"),
        col("dimension.EffectiveDate"),
        col("dimension.ExpirationDate"),
        col("dimension.IsCurrent")
    )
    .orderBy(
        col("dimension.PlateNo"),
        col("dimension.EffectiveDate")
    )
)

In [0]:
duplicate_current_cars_df = (
    final_dim_car_df
    .filter(col("IsCurrent") == True)
    .groupBy("PlateNo")
    .count()
    .filter(col("count") > 1)
)

duplicate_current_count = (
    duplicate_current_cars_df.count()
)

print(
    "PlateNo values with multiple current versions:",
    duplicate_current_count
)

display(duplicate_current_cars_df)


In [0]:
changed_day2_cars_df = (
    cars_comparison_df
    .filter(col("ChangeType") == "CHANGED")
)

print(
    "Changed existing Cars:",
    changed_day2_cars_df.count()
)

display(
    changed_day2_cars_df.orderBy("PlateNo")
)

In [0]:
missing_current_cars_df = (
    day2_cars
    .select("PlateNo")
    .distinct()
    .join(
        final_dim_car_df
        .filter(col("IsCurrent") == True)
        .select("PlateNo")
        .distinct(),
        "PlateNo",
        "left_anti"
    )
)

print(
    "Day 2 Cars missing a current Gold version:",
    missing_current_cars_df.count()
)

display(missing_current_cars_df)

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    coalesce,
    current_timestamp,
    row_number
)

from pyspark.sql.window import Window

day1_customers = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/customers")
)

day2_customers = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/customers")
)

print("Day 1 Customers:", day1_customers.count())
print("Day 2 Customers:", day2_customers.count())

display(day1_customers.orderBy("CustomerNo"))
display(day2_customers.orderBy("CustomerNo"))

In [0]:
print("Day 1 duplicate CustomerNo values:")

day1_customers.groupBy("CustomerNo") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("Day 2 duplicate CustomerNo values:")

day2_customers.groupBy("CustomerNo") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
customer_attributes = [
    "CustomerName",
    "Address",
    "DriversLicenseNo",
    "Age",
    "Email",
    "MobileNo"
]

def add_customer_hash(dataframe):
    return dataframe.withColumn(
        "AttributeHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in customer_attributes
                ]
            ),
            256
        )
    )

day1_customers_hashed = add_customer_hash(
    day1_customers
)

day2_customers_hashed = add_customer_hash(
    day2_customers
)

In [0]:
from pyspark.sql.functions import when

customers_comparison_df = (
    day2_customers_hashed.alias("day2")
    .join(
        day1_customers_hashed.alias("day1"),
        col("day2.CustomerNo")
        == col("day1.CustomerNo"),
        "left"
    )
    .select(
        col("day2.CustomerNo").alias("CustomerNo"),
        col("day2.CustomerName").alias("CustomerName"),

        col("day1.Address").alias("Day1Address"),
        col("day2.Address").alias("Day2Address"),

        col("day1.Email").alias("Day1Email"),
        col("day2.Email").alias("Day2Email"),

        col("day1.MobileNo").alias("Day1MobileNo"),
        col("day2.MobileNo").alias("Day2MobileNo"),

        when(
            col("day1.CustomerNo").isNull(),
            lit("NEW")
        )
        .when(
            col("day1.AttributeHash")
            != col("day2.AttributeHash"),
            lit("CHANGED")
        )
        .otherwise(
            lit("UNCHANGED")
        )
        .alias("ChangeType")
    )
)

display(
    customers_comparison_df.orderBy(
        "ChangeType",
        "CustomerNo"
    )
)

In [0]:
customers_comparison_df.groupBy(
    "ChangeType"
).count().orderBy(
    "ChangeType"
).show()

In [0]:
display(
    customers_comparison_df
    .filter(
        col("ChangeType").isin(
            "NEW",
            "CHANGED"
        )
    )
    .orderBy("CustomerNo")
)

In [0]:
dim_customer_path = (
    f"{gold_base}/dim_customer"
)

print(
    "dim_customer path:",
    dim_customer_path
)

In [0]:
day1_customer_dimension_df = (
    day1_customers_hashed
    .withColumn(
        "CustomerKey",
        row_number().over(
            Window.orderBy("CustomerNo")
        )
    )
    .withColumn(
        "EffectiveDate",
        current_timestamp()
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "CustomerKey",
        "CustomerNo",
        "CustomerName",
        "Address",
        "DriversLicenseNo",
        "Age",
        "Email",
        "MobileNo",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(
    day1_customer_dimension_df.orderBy(
        "CustomerNo"
    )
)

In [0]:
day1_customer_dimension_df.printSchema()

print(
    "Initial dim_customer row count:",
    day1_customer_dimension_df.count()
)

day1_customer_dimension_df.groupBy(
    "IsCurrent"
).count().show()

In [0]:
day1_customer_dimension_df.groupBy(
    "CustomerNo"
).count().filter(
    col("count") > 1
).show()

In [0]:
from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark, dim_car_path):
    (
        day1_car_dimension_df.write
        .format("delta")
        .mode("overwrite")
        .save(dim_car_path)
    )

    print("Initial dim_car created.")
else:
    print("dim_car already exists. Initialization skipped.")

In [0]:
current_gold_customers_df = (
    spark.read
    .format("delta")
    .load(dim_customer_path)
    .filter(col("IsCurrent") == True)
)

day2_customer_changes_df = (
    day2_customers_hashed.alias("source")
    .join(
        current_gold_customers_df.alias("target"),
        col("source.CustomerNo")
        == col("target.CustomerNo"),
        "left"
    )
    .filter(
        col("target.CustomerNo").isNull()
        |
        (
            col("source.AttributeHash")
            != col("target.AttributeHash")
        )
    )
    .select("source.*")
)

display(
    day2_customer_changes_df.orderBy(
        "CustomerNo"
    )
)

print(
    "Day 2 new or changed Customers:",
    day2_customer_changes_df.count()
)

In [0]:
customer_day2_timestamp = (
    spark.sql(
        "SELECT current_timestamp() AS ProcessingTimestamp"
    )
    .first()["ProcessingTimestamp"]
)

print(
    "Customer Day 2 processing timestamp:",
    customer_day2_timestamp
)

In [0]:
from delta.tables import DeltaTable

dim_customer_delta = DeltaTable.forPath(
    spark,
    dim_customer_path
)

(
    dim_customer_delta.alias("target")
    .merge(
        day2_customers_hashed.alias("source"),
        """
        target.CustomerNo = source.CustomerNo
        AND target.IsCurrent = true
        """
    )
    .whenMatchedUpdate(
        condition="""
        target.AttributeHash <> source.AttributeHash
        """,
        set={
            "ExpirationDate":
                lit(customer_day2_timestamp),

            "IsCurrent":
                lit(False)
        }
    )
    .execute()
)

print(
    "Changed Day 1 Customer versions were expired."
)

In [0]:
customer_gold_after_expiry_df = (
    spark.read
    .format("delta")
    .load(dim_customer_path)
)

current_max_customer_key = (
    customer_gold_after_expiry_df
    .selectExpr(
        "COALESCE(MAX(CustomerKey), 0) AS MaxCustomerKey"
    )
    .first()["MaxCustomerKey"]
)

print(
    "Current maximum CustomerKey:",
    current_max_customer_key
)

In [0]:
day2_customer_new_versions_df = (
    day2_customer_changes_df
    .withColumn(
        "CustomerKey",
        row_number().over(
            Window.orderBy("CustomerNo")
        ) + lit(current_max_customer_key)
    )
    .withColumn(
        "EffectiveDate",
        lit(customer_day2_timestamp).cast("timestamp")
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "CustomerKey",
        "CustomerNo",
        "CustomerName",
        "Address",
        "DriversLicenseNo",
        "Age",
        "Email",
        "MobileNo",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(
    day2_customer_new_versions_df.orderBy(
        "CustomerNo"
    )
)

In [0]:
customer_version_count = (
    day2_customer_new_versions_df.count()
)

if customer_version_count > 0:
    (
        day2_customer_new_versions_df.write
        .format("delta")
        .mode("append")
        .save(dim_customer_path)
    )

    print(
        "Day 2 Customer versions inserted:",
        customer_version_count
    )
else:
    print(
        "No new or changed Day 2 Customers were found."
    )

In [0]:
final_dim_customer_df = (
    spark.read
    .format("delta")
    .load(dim_customer_path)
)

display(
    final_dim_customer_df.orderBy(
        "CustomerNo",
        "EffectiveDate"
    )
)

In [0]:
final_dim_customer_df.groupBy(
    "IsCurrent"
).count().show()

In [0]:
customers_with_history_df = (
    final_dim_customer_df
    .groupBy("CustomerNo")
    .count()
    .filter(col("count") > 1)
)

display(
    customers_with_history_df.orderBy(
        "CustomerNo"
    )
)

In [0]:
duplicate_current_customers_df = (
    final_dim_customer_df
    .filter(col("IsCurrent") == True)
    .groupBy("CustomerNo")
    .count()
    .filter(col("count") > 1)
)

print(
    "CustomerNo values with multiple current versions:",
    duplicate_current_customers_df.count()
)

display(duplicate_current_customers_df)

In [0]:
missing_current_customers_df = (
    day2_customers
    .select("CustomerNo")
    .distinct()
    .join(
        final_dim_customer_df
        .filter(col("IsCurrent") == True)
        .select("CustomerNo")
        .distinct(),
        "CustomerNo",
        "left_anti"
    )
)

print(
    "Day 2 Customers missing a current Gold version:",
    missing_current_customers_df.count()
)

display(missing_current_customers_df)

In [0]:
day1_employees = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/employees")
)

day2_employees = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/employees")
)

print("Day 1 Employees:", day1_employees.count())
print("Day 2 Employees:", day2_employees.count())

display(day1_employees.orderBy("EmployeeNo"))
display(day2_employees.orderBy("EmployeeNo"))

In [0]:
employee_attributes = [
    "EmployeeName",
    "Address",
    "BirthDate",
    "Email",
    "Phone"
]

def add_employee_hash(dataframe):
    return dataframe.withColumn(
        "AttributeHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                      lit("")
                  )
                    for column_name in employee_attributes
                ]
            ),
           256
        )
    )

day1_employees_hashed = add_employee_hash(day1_employees)
day2_employees_hashed = add_employee_hash(day2_employees)


In [0]:
from pyspark.sql.functions import col, lit, when

employees_comparison_df = (
    day2_employees_hashed.alias("day2")
    .join(
        day1_employees_hashed.alias("day1"),
        col("day2.EmployeeNo") == col("day1.EmployeeNo"),
        "left"
    )
    .select(
        col("day2.EmployeeNo").alias("EmployeeNo"),
        col("day2.EmployeeName").alias("EmployeeName"),
        col("day1.Address").alias("Day1Address"),
        col("day2.Address").alias("Day2Address"),
        col("day1.Email").alias("Day1Email"),
        col("day2.Email").alias("Day2Email"),
        col("day1.Phone").alias("Day1Phone"),
        col("day2.Phone").alias("Day2Phone"),
        when(
            col("day1.EmployeeNo").isNull(),
            lit("NEW")
        )
        .when(
            col("day1.AttributeHash") != col("day2.AttributeHash"),
            lit("CHANGED")
        )
        .otherwise(
            lit("UNCHANGED")
        )
        .alias("ChangeType")
    )
)

display(
    employees_comparison_df.orderBy(
        "ChangeType",
        "EmployeeNo"
    )
)

employees_comparison_df.groupBy(
    "ChangeType"
).count().show()

In [0]:
from pyspark.sql.functions import (
    lit,
    current_timestamp,
    row_number
)

from pyspark.sql.window import Window

dim_employee_path = f"{gold_base}/dim_employee"

day1_employee_dimension_df = (
    day1_employees_hashed
    .withColumn(
        "EmployeeKey",
        row_number().over(
            Window.orderBy("EmployeeNo")
        )
    )
    .withColumn(
        "EffectiveDate",
        current_timestamp()
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "EmployeeKey",
        "EmployeeNo",
        "EmployeeName",
        "Address",
        "BirthDate",
        "Email",
        "Phone",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(
    day1_employee_dimension_df.orderBy("EmployeeNo")
)

In [0]:
from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark, dim_car_path):
    (
        day1_car_dimension_df.write
        .format("delta")
        .mode("overwrite")
        .save(dim_car_path)
    )

    print("Initial dim_car created.")
else:
    print("dim_car already exists. Initialization skipped.")

In [0]:
current_gold_employees_df = (
    spark.read
    .format("delta")
    .load(dim_employee_path)
    .filter(col("IsCurrent") == True)
)

day2_employee_changes_df = (
    day2_employees_hashed.alias("source")
    .join(
        current_gold_employees_df.alias("target"),
        col("source.EmployeeNo") == col("target.EmployeeNo"),
        "left"
    )
    .filter(
        col("target.EmployeeNo").isNull()
        |
        (
            col("source.AttributeHash")
            != col("target.AttributeHash")
        )
    )
    .select("source.*")
)

display(day2_employee_changes_df.orderBy("EmployeeNo"))

print(
    "Day 2 new or changed Employees:",
    day2_employee_changes_df.count()
)

In [0]:
employee_day2_timestamp = (
    spark.sql(
        "SELECT current_timestamp() AS ProcessingTimestamp"
    )
    .first()["ProcessingTimestamp"]
)

print(
    "Employee Day 2 processing timestamp:",
    employee_day2_timestamp
)

In [0]:
dim_employee_delta = DeltaTable.forPath(
    spark,
    dim_employee_path
)

(
    dim_employee_delta.alias("target")
    .merge(
        day2_employees_hashed.alias("source"),
        """
        target.EmployeeNo = source.EmployeeNo
        AND target.IsCurrent = true
        """
    )
    .whenMatchedUpdate(
        condition="""
        target.AttributeHash <> source.AttributeHash
        """,
        set={
            "ExpirationDate": lit(employee_day2_timestamp),
            "IsCurrent": lit(False)
        }
    )
    .execute()
)

print("Changed Day 1 Employee versions were expired.")

In [0]:
employee_gold_after_expiry_df = (
    spark.read
    .format("delta")
    .load(dim_employee_path)
)

current_max_employee_key = (
    employee_gold_after_expiry_df
    .selectExpr(
        "COALESCE(MAX(EmployeeKey), 0) AS MaxEmployeeKey"
    )
    .first()["MaxEmployeeKey"]
)

print(
    "Current maximum EmployeeKey:",
    current_max_employee_key
)

In [0]:
day2_employee_new_versions_df = (
    day2_employee_changes_df
    .withColumn(
        "EmployeeKey",
        row_number().over(
            Window.orderBy("EmployeeNo")
        ) + lit(current_max_employee_key)
    )
    .withColumn(
        "EffectiveDate",
        lit(employee_day2_timestamp).cast("timestamp")
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "EmployeeKey",
        "EmployeeNo",
        "EmployeeName",
        "Address",
        "BirthDate",
        "Email",
        "Phone",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(day2_employee_new_versions_df)

In [0]:
employee_version_count = (
    day2_employee_new_versions_df.count()
)

if employee_version_count > 0:
    (
        day2_employee_new_versions_df.write
        .format("delta")
        .mode("append")
        .save(dim_employee_path)
    )

    print(
        "Day 2 Employee versions inserted:",
        employee_version_count
    )
else:
    print(
        "No new or changed Day 2 Employees were found."
    )

In [0]:
final_dim_employee_df = (
    spark.read
    .format("delta")
    .load(dim_employee_path)
)

display(
    final_dim_employee_df.orderBy(
        "EmployeeNo",
        "EffectiveDate"
    )
)

In [0]:
duplicate_current_employees_df = (
    final_dim_employee_df
    .filter(col("IsCurrent") == True)
    .groupBy("EmployeeNo")
    .count()
    .filter(col("count") > 1)
)

print(
    "EmployeeNo values with multiple current versions:",
    duplicate_current_employees_df.count()
)

In [0]:
missing_current_employees_df = (
    day2_employees
    .select("EmployeeNo")
    .distinct()
    .join(
        final_dim_employee_df
        .filter(col("IsCurrent") == True)
        .select("EmployeeNo")
        .distinct(),
        "EmployeeNo",
        "left_anti"
    )
)

print(
    "Day 2 Employees missing a current Gold version:",
    missing_current_employees_df.count()
)

display(missing_current_employees_df)

In [0]:
missing_current_employees_df = (
    day2_employees
    .select("EmployeeNo")
    .distinct()
    .join(
        final_dim_employee_df
        .filter(col("IsCurrent") == True)
        .select("EmployeeNo")
        .distinct(),
        "EmployeeNo",
        "left_anti"
    )
)

print(
    "Day 2 Employees missing a current Gold version:",
    missing_current_employees_df.count()
)

display(missing_current_employees_df)

In [0]:
day1_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/locations")
)

day2_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/locations")
)

print("Day 1 Locations:", day1_locations.count())
print("Day 2 Locations:", day2_locations.count())

display(day1_locations.orderBy("LocationCode"))
display(day2_locations.orderBy("LocationCode"))

In [0]:
day1_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/locations")
)

day2_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/locations")
)

print("Day 1 Locations:", day1_locations.count())
print("Day 2 Locations:", day2_locations.count())

display(day1_locations.orderBy("LocationCode"))
display(day2_locations.orderBy("LocationCode"))

In [0]:
silver_base = "abfss://autorent@stautorentjs2026.dfs.core.windows.net/silver"

day1_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/locations")
)

day2_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/locations")
)

print(
    "Day 1 Locations:",
    day1_locations.count()
)

print(
    "Day 2 Locations:",
    day2_locations.count()
)

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    coalesce
)

location_attributes = [
    "Address1",
    "Address2",
    "City",
    "Province"
]

def add_location_hash(dataframe):
    return dataframe.withColumn(
        "AttributeHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in location_attributes
                ]
            ),
            256
        )
    )

day1_locations_hashed = add_location_hash(
    day1_locations
)

day2_locations_hashed = add_location_hash(
    day2_locations
)

print(
    "Day 1 hashed Location rows:",
    day1_locations_hashed.count()
)

print(
    "Day 2 hashed Location rows:",
    day2_locations_hashed.count()
)

In [0]:
from pyspark.sql.functions import col, lit, when

locations_comparison_df = (
    day2_locations_hashed.alias("day2")
    .join(
        day1_locations_hashed.alias("day1"),
        col("day2.LocationCode")
        == col("day1.LocationCode"),
        "left"
    )
    .select(
        col("day2.LocationCode").alias("LocationCode"),
        col("day1.Address1").alias("Day1Address1"),
        col("day2.Address1").alias("Day2Address1"),
        col("day1.Address2").alias("Day1Address2"),
        col("day2.Address2").alias("Day2Address2"),
        col("day1.City").alias("Day1City"),
        col("day2.City").alias("Day2City"),
        col("day1.Province").alias("Day1Province"),
        col("day2.Province").alias("Day2Province"),
        when(
            col("day1.LocationCode").isNull(),
            lit("NEW")
        )
        .when(
            col("day1.AttributeHash")
            != col("day2.AttributeHash"),
            lit("CHANGED")
        )
        .otherwise(
            lit("UNCHANGED")
        )
        .alias("ChangeType")
    )
)

display(
    locations_comparison_df.orderBy(
        "ChangeType",
        "LocationCode"
    )
)

locations_comparison_df.groupBy(
    "ChangeType"
).count().show()

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    coalesce
)

location_attributes = [
    "Address1",
    "Address2",
    "City",
    "Province"
]

def add_location_hash(dataframe):
    return dataframe.withColumn(
        "AttributeHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in location_attributes
                ]
            ),
            256
        )
    )

day1_locations_hashed = add_location_hash(
    day1_locations
)

day2_locations_hashed = add_location_hash(
    day2_locations
)

print(
    "Day 1 hashed Location rows:",
    day1_locations_hashed.count()
)

print(
    "Day 2 hashed Location rows:",
    day2_locations_hashed.count()
)

display(
    day2_locations_hashed.orderBy("LocationCode")
)

In [0]:
dim_location_path = f"{gold_base}/dim_location"

day1_location_dimension_df = (
    day1_locations_hashed
    .withColumn(
        "LocationKey",
        row_number().over(
            Window.orderBy("LocationCode")
        )
    )
    .withColumn(
        "EffectiveDate",
        current_timestamp()
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "LocationKey",
        "LocationCode",
        "Address1",
        "Address2",
        "City",
        "Province",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(
    day1_location_dimension_df.orderBy("LocationCode")
)

In [0]:
from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark, dim_car_path):
    (
        day1_car_dimension_df.write
        .format("delta")
        .mode("overwrite")
        .save(dim_car_path)
    )

    print("Initial dim_car created.")
else:
    print("dim_car already exists. Initialization skipped.")

In [0]:
current_gold_locations_df = (
    spark.read
    .format("delta")
    .load(dim_location_path)
    .filter(col("IsCurrent") == True)
)

day2_location_changes_df = (
    day2_locations_hashed.alias("source")
    .join(
        current_gold_locations_df.alias("target"),
        col("source.LocationCode") == col("target.LocationCode"),
        "left"
    )
    .filter(
        col("target.LocationCode").isNull()
        |
        (
            col("source.AttributeHash") != col("target.AttributeHash")
        )
    )
    .select("source.*")
)

display(
    day2_location_changes_df.orderBy("LocationCode")
)

print(
    "Day 2 new or changed Locations:",
    day2_location_changes_df.count()
)


In [0]:
location_day2_timestamp = (
    spark.sql(
        "SELECT current_timestamp() AS ProcessingTimestamp"
    )
    .first()["ProcessingTimestamp"]
)

print(
    "Location Day 2 processing timestamp:",
    location_day2_timestamp
)

In [0]:
location_day2_timestamp = (
    spark.sql(
        "SELECT current_timestamp() AS ProcessingTimestamp"
    )
    .first()["ProcessingTimestamp"]
)

print(
    "Location Day 2 processing timestamp:",
    location_day2_timestamp
)

In [0]:
location_gold_after_expiry_df = (
    spark.read
    .format("delta")
    .load(dim_location_path)
)

current_max_location_key = (
    location_gold_after_expiry_df
    .selectExpr(
        "COALESCE(MAX(LocationKey), 0) AS MaxLocationKey"
    )
    .first()["MaxLocationKey"]
)

print(
    "Current maximum LocationKey:",
    current_max_location_key
)

In [0]:
day2_location_new_versions_df = (
    day2_location_changes_df
    .withColumn(
        "LocationKey",
        row_number().over(
            Window.orderBy("LocationCode")
        ) + lit(current_max_location_key)
    )
    .withColumn(
        "EffectiveDate",
        lit(location_day2_timestamp).cast("timestamp")
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "LocationKey",
        "LocationCode",
        "Address1",
        "Address2",
        "City",
        "Province",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(
    day2_location_new_versions_df.orderBy("LocationCode")
)

In [0]:
day2_location_new_versions_df = (
    day2_location_changes_df
    .withColumn(
        "LocationKey",
        row_number().over(
            Window.orderBy("LocationCode")
        ) + lit(current_max_location_key)
    )
    .withColumn(
        "EffectiveDate",
        lit(location_day2_timestamp).cast("timestamp")
    )
    .withColumn(
        "ExpirationDate",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
    .select(
        "LocationKey",
        "LocationCode",
        "Address1",
        "Address2",
        "City",
        "Province",
        "AttributeHash",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

display(
    day2_location_new_versions_df.orderBy("LocationCode")
)

In [0]:
final_dim_location_df = (
    spark.read
    .format("delta")
    .load(dim_location_path)
)

display(
    final_dim_location_df.orderBy(
        "LocationCode",
        "EffectiveDate"
    )
)

In [0]:
missing_current_locations_df = (
    day2_locations
    .select("LocationCode")
    .distinct()
    .join(
        final_dim_location_df
        .filter(col("IsCurrent") == True)
        .select("LocationCode")
        .distinct(),
        "LocationCode",
        "left_anti"
    )
)

print(
    "Day 2 Locations missing a current Gold version:",
    missing_current_locations_df.count()
)

display(missing_current_locations_df)

In [0]:
duplicate_current_locations_df = (
    final_dim_location_df
    .filter(col("IsCurrent") == True)
    .groupBy("LocationCode")
    .count()
    .filter(col("count") > 1)
)

print(
    "LocationCode values with multiple current versions:",
    duplicate_current_locations_df.count()
)

In [0]:
day3_base = f"{bronze_base}/day3"
day3_silver_base = f"{silver_base}/day3"

print("Day 3 Bronze:", day3_base)
print("Day 3 Silver:", day3_silver_base)


In [0]:
from pyspark.sql.functions import col, trim

day3_cars_source = f"{day3_base}/Cars.txt"
day3_cars_silver = f"{day3_silver_base}/cars"

day3_cars_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day3_cars_source)
)

day3_cars_clean_df = (
    day3_cars_df
    .select(
        trim(col("`PlateNo`")).alias("PlateNo"),
        trim(col("`Car Name`")).alias("CarName"),
        trim(col("`Manufacturer`")).alias("Manufacturer"),
        col("`Year Model`").cast("int").alias("YearModel"),
        trim(col("`Car Body Type`")).alias("CarBodyType"),
        trim(col("`Transmission`")).alias("Transmission"),
        col("`Daily Rate (in Peso)`")
            .cast("decimal(10,2)")
            .alias("DailyRate")
    )
)

display(day3_cars_clean_df)

In [0]:
day3_cars_clean_df.printSchema()

print(
    "Day 3 Cars row count:",
    day3_cars_clean_df.count()
)

print(
    "Day 3 duplicate PlateNo values:",
    day3_cars_clean_df
        .groupBy("PlateNo")
        .count()
        .filter(col("count") > 1)
        .count()
)

(
    day3_cars_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day3_cars_silver)
)

print("Day 3 Cars successfully written to Silver.")

In [0]:
day3_customers_source = f"{day3_base}/Customers.txt"
day3_customers_silver = f"{day3_silver_base}/customers"

day3_customers_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day3_customers_source)
)

day3_customers_clean_df = (
    day3_customers_df
    .select(
        col("`CustomerNo`").cast("int").alias("CustomerNo"),
        trim(col("`Name`")).alias("CustomerName"),
        trim(col("`Address`")).alias("Address"),
        trim(
            col("`Driver's License No.`")
        ).alias("DriversLicenseNo"),
        col("`Age`").cast("int").alias("Age"),
        trim(col("`Email`")).alias("Email"),
        trim(col("`Mobile No.`")).alias("MobileNo")
    )
)

display(day3_customers_clean_df)

In [0]:
day3_customers_clean_df.printSchema()

print(
    "Day 3 Customers row count:",
    day3_customers_clean_df.count()
)

print(
    "Day 3 duplicate CustomerNo values:",
    day3_customers_clean_df
        .groupBy("CustomerNo")
        .count()
        .filter(col("count") > 1)
        .count()
)

(
    day3_customers_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day3_customers_silver)
)

print("Day 3 Customers successfully written to Silver.")


In [0]:
from pyspark.sql.functions import to_date, coalesce

day3_employees_source = f"{day3_base}/Employees.txt"
day3_employees_silver = f"{day3_silver_base}/employees"

day3_employees_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day3_employees_source)
)

day3_employees_clean_df = (
    day3_employees_df
    .select(
        col("`EmployeeNo`").cast("int").alias("EmployeeNo"),
        trim(col("`Name`")).alias("EmployeeName"),
        trim(col("`Address`")).alias("Address"),

        coalesce(
            to_date(col("`Birthdate`"), "yyyy-MM-dd"),
            to_date(col("`Birthdate`"), "M/d/yyyy"),
            to_date(col("`Birthdate`"), "MM/dd/yyyy")
        ).alias("BirthDate"),

        trim(col("`Email`")).alias("Email"),
        trim(col("`Phone`")).alias("Phone")
    )
)

display(day3_employees_clean_df)

In [0]:
day3_employees_clean_df.printSchema()

print(
    "Day 3 Employees row count:",
    day3_employees_clean_df.count()
)

print(
    "Employees with unparsed BirthDate:",
    day3_employees_clean_df
        .filter(col("BirthDate").isNull())
        .count()
)

print(
    "Day 3 duplicate EmployeeNo values:",
    day3_employees_clean_df
        .groupBy("EmployeeNo")
        .count()
        .filter(col("count") > 1)
        .count()
)

(
    day3_employees_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day3_employees_silver)
)

print("Day 3 Employees successfully written to Silver.")

In [0]:
day3_locations_source = f"{day3_base}/Location.txt"
day3_locations_silver = f"{day3_silver_base}/locations"

day3_locations_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day3_locations_source)
)

day3_locations_clean_df = (
    day3_locations_df
    .select(
        trim(col("`LocationCode`")).alias("LocationCode"),
        trim(col("`Address1`")).alias("Address1"),
        trim(col("`Address2`")).alias("Address2"),
        trim(col("`City`")).alias("City"),
        trim(col("`Province`")).alias("Province")
    )
)

display(day3_locations_clean_df)


In [0]:
day3_locations_clean_df.printSchema()

print(
    "Day 3 Locations row count:",
    day3_locations_clean_df.count()
)

print(
    "Day 3 duplicate LocationCode values:",
    day3_locations_clean_df
        .groupBy("LocationCode")
        .count()
        .filter(col("count") > 1)
        .count()
)

(
    day3_locations_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day3_locations_silver)
)

print("Day 3 Locations successfully written to Silver.")


In [0]:
day3_transactions_source = (
    f"{day3_base}/Transactions.txt"
)

day3_transactions_silver = (
    f"{day3_silver_base}/transactions"
)

day3_transactions_df = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(day3_transactions_source)
)

display(day3_transactions_df)

In [0]:
from pyspark.sql.functions import (
    try_to_timestamp,
    coalesce,
    lit
)

day3_transactions_clean_df = (
    day3_transactions_df
    .select(
        trim(col("`plateno`")).alias("PlateNo"),
        trim(col("`locationcode`")).alias("LocationCode"),
        col("`customerno`").cast("int").alias("CustomerNo"),
        col("`employeeno`").cast("int").alias("EmployeeNo"),

        coalesce(
            try_to_timestamp(
                col("`date`"),
                lit("M/d/yyyy H:mm")
            ),
            try_to_timestamp(
                col("`date`"),
                lit("M/d/yyyy H:mm:ss")
            ),
            try_to_timestamp(
                col("`date`"),
                lit("MM/dd/yyyy HH:mm")
            ),
            try_to_timestamp(
                col("`date`"),
                lit("yyyy-MM-dd HH:mm:ss.SSS")
            )
        ).alias("TransactionDate"),

        col("`qty`").cast("int").alias("Quantity"),

        col("`amount`")
            .cast("decimal(12,2)")
            .alias("Amount"),

        trim(col("`type`")).alias("TransactionType"),

        col("`days`").cast("int").alias("Days")
    )
)

display(day3_transactions_clean_df)

In [0]:
day3_transactions_clean_df.printSchema()

print(
    "Day 3 Transactions row count:",
    day3_transactions_clean_df.count()
)

print(
    "Invalid or unparsed Day 3 transaction dates:",
    day3_transactions_clean_df
        .filter(col("TransactionDate").isNull())
        .count()
)

day3_transactions_clean_df.groupBy(
    "TransactionType"
).count().show()

In [0]:
(
    day3_transactions_clean_df.write
    .format("delta")
    .mode("overwrite")
    .save(day3_transactions_silver)
)

print("Day 3 Transactions successfully written to Silver.")

In [0]:
day3_silver_paths = {
    "cars": f"{day3_silver_base}/cars",
    "customers": f"{day3_silver_base}/customers",
    "employees": f"{day3_silver_base}/employees",
    "locations": f"{day3_silver_base}/locations",
    "transactions": f"{day3_silver_base}/transactions"
}

for dataset_name, dataset_path in day3_silver_paths.items():
    dataset_count = (
        spark.read
        .format("delta")
        .load(dataset_path)
        .count()
    )

    print(
        dataset_name,
        "rows:",
        dataset_count
    )

In [0]:
from delta.tables import DeltaTable

from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    coalesce,
    row_number
)

from pyspark.sql.window import Window

In [0]:
def apply_scd_type_2(
    source_df,
    target_path,
    business_key,
    surrogate_key,
    tracked_columns,
    output_columns,
    batch_name
):
    """
    Applies an SCD Type 2 load to an existing Delta dimension.

    source_df:
        Cleaned source snapshot.

    target_path:
        Existing Gold Delta dimension path.

    business_key:
        Source business identifier.

    surrogate_key:
        Warehouse-generated dimension key.

    tracked_columns:
        Attributes used for change detection.

    output_columns:
        Dimension attribute columns to retain.

    batch_name:
        Text used in validation messages.
    """

    source_hashed_df = (
        source_df
        .dropDuplicates([business_key])
        .withColumn(
            "AttributeHash",
            sha2(
                concat_ws(
                    "||",
                    *[
                        coalesce(
                            col(column_name).cast("string"),
                            lit("")
                        )
                        for column_name in tracked_columns
                    ]
                ),
                256
            )
        )
    )

    target_current_df = (
        spark.read
        .format("delta")
        .load(target_path)
        .filter(col("IsCurrent") == True)
    )

    changes_df = (
        source_hashed_df.alias("source")
        .join(
            target_current_df.alias("target"),
            col(f"source.{business_key}")
            == col(f"target.{business_key}"),
            "left"
        )
        .filter(
            col(f"target.{business_key}").isNull()
            |
            (
                col("source.AttributeHash")
                != col("target.AttributeHash")
            )
        )
        .select("source.*")
    )

    change_count = changes_df.count()

    print(
        batch_name,
        "new or changed records:",
        change_count
    )

    if change_count == 0:
        print(
            batch_name,
            "requires no SCD Type 2 changes."
        )

        return (
            spark.read
            .format("delta")
            .load(target_path)
        )

    processing_timestamp = (
        spark.sql(
            "SELECT current_timestamp() AS ProcessingTimestamp"
        )
        .first()["ProcessingTimestamp"]
    )

    target_delta = DeltaTable.forPath(
        spark,
        target_path
    )

    (
        target_delta.alias("target")
        .merge(
            source_hashed_df.alias("source"),
            f"""
            target.{business_key} = source.{business_key}
            AND target.IsCurrent = true
            """
        )
        .whenMatchedUpdate(
            condition="""
            target.AttributeHash <> source.AttributeHash
            """,
            set={
                "ExpirationDate":
                    lit(processing_timestamp),

                "IsCurrent":
                    lit(False)
            }
        )
        .execute()
    )

    target_after_expiry_df = (
        spark.read
        .format("delta")
        .load(target_path)
    )

    current_max_key = (
        target_after_expiry_df
        .selectExpr(
            f"""
            COALESCE(
                MAX({surrogate_key}),
                0
            ) AS MaximumKey
            """
        )
        .first()["MaximumKey"]
    )

    new_versions_df = (
        changes_df
        .withColumn(
            surrogate_key,
            row_number().over(
                Window.orderBy(business_key)
            ) + lit(current_max_key)
        )
        .withColumn(
            "EffectiveDate",
            lit(processing_timestamp).cast("timestamp")
        )
        .withColumn(
            "ExpirationDate",
            lit(None).cast("timestamp")
        )
        .withColumn(
            "IsCurrent",
            lit(True)
        )
        .select(
            surrogate_key,
            business_key,
            *output_columns,
            "AttributeHash",
            "EffectiveDate",
            "ExpirationDate",
            "IsCurrent"
        )
    )

    (
        new_versions_df.write
        .format("delta")
        .mode("append")
        .save(target_path)
    )

    print(
        batch_name,
        "versions inserted:",
        new_versions_df.count()
    )

    final_dimension_df = (
        spark.read
        .format("delta")
        .load(target_path)
    )

    duplicate_current_count = (
        final_dimension_df
        .filter(col("IsCurrent") == True)
        .groupBy(business_key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    missing_current_count = (
        source_df
        .select(business_key)
        .distinct()
        .join(
            final_dimension_df
            .filter(col("IsCurrent") == True)
            .select(business_key)
            .distinct(),
            business_key,
            "left_anti"
        )
        .count()
    )

    print(
        batch_name,
        "business keys with multiple current versions:",
        duplicate_current_count
    )

    print(
        batch_name,
        "source records missing a current Gold version:",
        missing_current_count
    )

    return final_dimension_df

In [0]:
final_dim_car_day3_df = apply_scd_type_2(
    source_df=day3_cars_clean_df,

    target_path=dim_car_path,

    business_key="PlateNo",

    surrogate_key="CarKey",

    tracked_columns=[
        "CarName",
        "Manufacturer",
        "YearModel",
        "CarBodyType",
        "Transmission",
        "DailyRate"
    ],

    output_columns=[
        "CarName",
        "Manufacturer",
        "YearModel",
        "CarBodyType",
        "Transmission",
        "DailyRate"
    ],

    batch_name="Day 3 dim_car"
)

In [0]:
display(
    final_dim_car_day3_df.orderBy(
        "PlateNo",
        "EffectiveDate"
    )
)

In [0]:
final_dim_customer_day3_df = apply_scd_type_2(
    source_df=day3_customers_clean_df,

    target_path=dim_customer_path,

    business_key="CustomerNo",

    surrogate_key="CustomerKey",

    tracked_columns=[
        "CustomerName",
        "Address",
        "DriversLicenseNo",
        "Age",
        "Email",
        "MobileNo"
    ],

    output_columns=[
        "CustomerName",
        "Address",
        "DriversLicenseNo",
        "Age",
        "Email",
        "MobileNo"
    ],

    batch_name="Day 3 dim_customer"
)

In [0]:
display(
    final_dim_customer_day3_df.orderBy(
        "CustomerNo",
        "EffectiveDate"
    )
)

In [0]:
final_dim_employee_day3_df = apply_scd_type_2(
    source_df=day3_employees_clean_df,

    target_path=dim_employee_path,

    business_key="EmployeeNo",

    surrogate_key="EmployeeKey",

    tracked_columns=[
        "EmployeeName",
        "Address",
        "BirthDate",
        "Email",
        "Phone"
    ],

    output_columns=[
        "EmployeeName",
        "Address",
        "BirthDate",
        "Email",
        "Phone"
    ],

    batch_name="Day 3 dim_employee"
)

In [0]:
display(
    final_dim_employee_day3_df.orderBy(
        "EmployeeNo",
        "EffectiveDate"
    )
)

In [0]:
final_dim_location_day3_df = apply_scd_type_2(
    source_df=day3_locations_clean_df,

    target_path=dim_location_path,

    business_key="LocationCode",

    surrogate_key="LocationKey",

    tracked_columns=[
        "Address1",
        "Address2",
        "City",
        "Province"
    ],

    output_columns=[
        "Address1",
        "Address2",
        "City",
        "Province"
    ],

    batch_name="Day 3 dim_location"
)


In [0]:
display(
    final_dim_location_day3_df.orderBy(
        "LocationCode",
        "EffectiveDate"
    )
)

In [0]:
dimension_checks = [
    (
        "dim_car",
        dim_car_path,
        "PlateNo"
    ),
    (
        "dim_customer",
        dim_customer_path,
        "CustomerNo"
    ),
    (
        "dim_employee",
        dim_employee_path,
        "EmployeeNo"
    ),
    (
        "dim_location",
        dim_location_path,
        "LocationCode"
    )
]

for dimension_name, dimension_path, business_key in dimension_checks:

    dimension_df = (
        spark.read
        .format("delta")
        .load(dimension_path)
    )

    total_count = dimension_df.count()

    current_count = (
        dimension_df
        .filter(col("IsCurrent") == True)
        .count()
    )

    historical_count = (
        dimension_df
        .filter(col("IsCurrent") == False)
        .count()
    )

    duplicate_current_count = (
        dimension_df
        .filter(col("IsCurrent") == True)
        .groupBy(business_key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    print("-----------------------------------")
    print("Dimension:", dimension_name)
    print("Total records:", total_count)
    print("Current records:", current_count)
    print("Historical records:", historical_count)
    print(
        "Duplicate current business keys:",
        duplicate_current_count
    )

In [0]:
def display_dimension_history(
    dimension_path,
    business_key
):
    dimension_df = (
        spark.read
        .format("delta")
        .load(dimension_path)
    )

    history_keys_df = (
        dimension_df
        .groupBy(business_key)
        .count()
        .filter(col("count") > 1)
        .select(business_key)
    )

    display(
        dimension_df
        .join(
            history_keys_df,
            business_key,
            "inner"
        )
        .orderBy(
            business_key,
            "EffectiveDate"
        )
    )

In [0]:
display_dimension_history(
    dim_customer_path,
    "CustomerNo"
)

In [0]:
display_dimension_history(
    dim_employee_path,
    "EmployeeNo"
)

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    date_format,
    sha2,
    concat_ws,
    coalesce
)

day1_transactions = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/transactions")
    .withColumn("SourceBatch", lit("Day1"))
)

day2_transactions = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/transactions")
    .withColumn("SourceBatch", lit("Day2"))
)

day3_transactions = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day3/transactions")
    .withColumn("SourceBatch", lit("Day3"))
)

print("Day 1 Transactions:", day1_transactions.count())
print("Day 2 Transactions:", day2_transactions.count())
print("Day 3 Transactions:", day3_transactions.count())

In [0]:
all_transactions_df = (
    day1_transactions
    .unionByName(day2_transactions)
    .unionByName(day3_transactions)
)

print(
    "Combined transaction rows before deduplication:",
    all_transactions_df.count()
)

display(
    all_transactions_df.orderBy(
        "TransactionDate",
        "PlateNo"
    )
)

In [0]:
transaction_hash_columns = [
    "PlateNo",
    "LocationCode",
    "CustomerNo",
    "EmployeeNo",
    "TransactionDate",
    "Quantity",
    "Amount",
    "TransactionType",
    "Days"
]

all_transactions_hashed_df = (
    all_transactions_df
    .withColumn(
        "TransactionHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in transaction_hash_columns
                ]
            ),
            256
        )
    )
)

In [0]:
deduplicated_transactions_df = (
    all_transactions_hashed_df
    .dropDuplicates(["TransactionHash"])
)

print(
    "Transaction rows after deduplication:",
    deduplicated_transactions_df.count()
)

print(
    "Duplicate transaction rows removed:",
    all_transactions_hashed_df.count()
    - deduplicated_transactions_df.count()
)

In [0]:
gold_dim_car_all_df = (
    spark.read
    .format("delta")
    .load(dim_car_path)
)

In [0]:
day1_cars_lookup_source = (
    add_car_hash(day1_cars)
    .withColumn("SourceBatch", lit("Day1"))
)

day2_cars_lookup_source = (
    add_car_hash(day2_cars)
    .withColumn("SourceBatch", lit("Day2"))
)

day3_cars_lookup_source = (
    add_car_hash(day3_cars_clean_df)
    .withColumn("SourceBatch", lit("Day3"))
)

all_car_snapshots_df = (
    day1_cars_lookup_source
    .unionByName(day2_cars_lookup_source)
    .unionByName(day3_cars_lookup_source)
)

In [0]:
car_lookup_df = (
    all_car_snapshots_df.alias("source")
    .join(
        gold_dim_car_all_df.alias("gold"),
        (
            col("source.PlateNo")
            == col("gold.PlateNo")
        )
        &
        (
            col("source.AttributeHash")
            == col("gold.AttributeHash")
        ),
        "left"
    )
    .select(
        col("source.SourceBatch"),
        col("source.PlateNo"),
        col("gold.CarKey")
    )
    .dropDuplicates(
        [
            "SourceBatch",
            "PlateNo"
        ]
    )
)

display(
    car_lookup_df.orderBy(
        "SourceBatch",
        "PlateNo"
    )
)

In [0]:
gold_dim_customer_all_df = (
    spark.read
    .format("delta")
    .load(dim_customer_path)
)

In [0]:
day1_customer_lookup_source = (
    add_customer_hash(day1_customers)
    .withColumn("SourceBatch", lit("Day1"))
)

day2_customer_lookup_source = (
    add_customer_hash(day2_customers)
    .withColumn("SourceBatch", lit("Day2"))
)

day3_customer_lookup_source = (
    add_customer_hash(day3_customers_clean_df)
    .withColumn("SourceBatch", lit("Day3"))
)

all_customer_snapshots_df = (
    day1_customer_lookup_source
    .unionByName(day2_customer_lookup_source)
    .unionByName(day3_customer_lookup_source)
)

customer_lookup_df = (
    all_customer_snapshots_df.alias("source")
    .join(
        gold_dim_customer_all_df.alias("gold"),
        (
            col("source.CustomerNo")
            == col("gold.CustomerNo")
        )
        &
        (
            col("source.AttributeHash")
            == col("gold.AttributeHash")
        ),
        "left"
    )
    .select(
        col("source.SourceBatch"),
        col("source.CustomerNo"),
        col("gold.CustomerKey")
    )
    .dropDuplicates(
        [
            "SourceBatch",
            "CustomerNo"
        ]
    )
)

display(
    customer_lookup_df.orderBy(
        "SourceBatch",
        "CustomerNo"
    )
)

In [0]:
gold_dim_employee_all_df = (
    spark.read
    .format("delta")
    .load(dim_employee_path)
)

In [0]:
day1_employee_lookup_source = (
    add_employee_hash(day1_employees)
    .withColumn("SourceBatch", lit("Day1"))
)

day2_employee_lookup_source = (
    add_employee_hash(day2_employees)
    .withColumn("SourceBatch", lit("Day2"))
)

day3_employee_lookup_source = (
    add_employee_hash(day3_employees_clean_df)
    .withColumn("SourceBatch", lit("Day3"))
)

all_employee_snapshots_df = (
    day1_employee_lookup_source
    .unionByName(day2_employee_lookup_source)
    .unionByName(day3_employee_lookup_source)
)

employee_lookup_df = (
    all_employee_snapshots_df.alias("source")
    .join(
        gold_dim_employee_all_df.alias("gold"),
        (
            col("source.EmployeeNo")
            == col("gold.EmployeeNo")
        )
        &
        (
            col("source.AttributeHash")
            == col("gold.AttributeHash")
        ),
        "left"
    )
    .select(
        col("source.SourceBatch"),
        col("source.EmployeeNo"),
        col("gold.EmployeeKey")
    )
    .dropDuplicates(
        [
            "SourceBatch",
            "EmployeeNo"
        ]
    )
)

display(
    employee_lookup_df.orderBy(
        "SourceBatch",
        "EmployeeNo"
    )
)

In [0]:
gold_dim_location_all_df = (
    spark.read
    .format("delta")
    .load(dim_location_path)
)

In [0]:
day1_location_lookup_source = (
    add_location_hash(day1_locations)
    .withColumn("SourceBatch", lit("Day1"))
)

day2_location_lookup_source = (
    add_location_hash(day2_locations)
    .withColumn("SourceBatch", lit("Day2"))
)

day3_location_lookup_source = (
    add_location_hash(day3_locations_clean_df)
    .withColumn("SourceBatch", lit("Day3"))
)

all_location_snapshots_df = (
    day1_location_lookup_source
    .unionByName(day2_location_lookup_source)
    .unionByName(day3_location_lookup_source)
)

location_lookup_df = (
    all_location_snapshots_df.alias("source")
    .join(
        gold_dim_location_all_df.alias("gold"),
        (
            col("source.LocationCode")
            == col("gold.LocationCode")
        )
        &
        (
            col("source.AttributeHash")
            == col("gold.AttributeHash")
        ),
        "left"
    )
    .select(
        col("source.SourceBatch"),
        col("source.LocationCode"),
        col("gold.LocationKey")
    )
    .dropDuplicates(
        [
            "SourceBatch",
            "LocationCode"
        ]
    )
)

display(
    location_lookup_df.orderBy(
        "SourceBatch",
        "LocationCode"
    )
)

In [0]:
transactions_alias = deduplicated_transactions_df.alias("fact")
cars_alias = car_lookup_df.alias("car")
customers_alias = customer_lookup_df.alias("customer")
employees_alias = employee_lookup_df.alias("employee")
locations_alias = location_lookup_df.alias("location")

In [0]:
resolved_fact_df = (
    transactions_alias

    .join(
        cars_alias,
        (
            col("fact.SourceBatch")
            == col("car.SourceBatch")
        )
        &
        (
            col("fact.PlateNo")
            == col("car.PlateNo")
        ),
        "left"
    )

    .join(
        customers_alias,
        (
            col("fact.SourceBatch")
            == col("customer.SourceBatch")
        )
        &
        (
            col("fact.CustomerNo")
            == col("customer.CustomerNo")
        ),
        "left"
    )

    .join(
        employees_alias,
        (
            col("fact.SourceBatch")
            == col("employee.SourceBatch")
        )
        &
        (
            col("fact.EmployeeNo")
            == col("employee.EmployeeNo")
        ),
        "left"
    )

    .join(
        locations_alias,
        (
            col("fact.SourceBatch")
            == col("location.SourceBatch")
        )
        &
        (
            col("fact.LocationCode")
            == col("location.LocationCode")
        ),
        "left"
    )

    .select(
        col("fact.TransactionHash"),
        col("car.CarKey"),
        col("customer.CustomerKey"),
        col("employee.EmployeeKey"),
        col("location.LocationKey"),

        date_format(
            col("fact.TransactionDate"),
            "yyyyMMdd"
        )
        .cast("int")
        .alias("DateKey"),

        col("fact.TransactionDate"),
        col("fact.Quantity"),
        col("fact.Amount"),
        col("fact.TransactionType"),
        col("fact.Days"),
        col("fact.SourceBatch")
    )
)

display(
    resolved_fact_df.orderBy(
        "TransactionDate",
        "CarKey"
    )
)

In [0]:
missing_car_keys = (
    resolved_fact_df
    .filter(col("CarKey").isNull())
    .count()
)

missing_customer_keys = (
    resolved_fact_df
    .filter(col("CustomerKey").isNull())
    .count()
)

missing_employee_keys = (
    resolved_fact_df
    .filter(col("EmployeeKey").isNull())
    .count()
)

missing_location_keys = (
    resolved_fact_df
    .filter(col("LocationKey").isNull())
    .count()
)

print("Missing CarKey values:", missing_car_keys)
print("Missing CustomerKey values:", missing_customer_keys)
print("Missing EmployeeKey values:", missing_employee_keys)
print("Missing LocationKey values:", missing_location_keys)

In [0]:
missing_dimension_records_df = (
    resolved_fact_df
    .filter(
        col("CarKey").isNull()
        |
        col("LocationKey").isNull()
    )
)

display(missing_dimension_records_df)

print(
    "Rows with unresolved dimension keys:",
    missing_dimension_records_df.count()
)

In [0]:
missing_car_references_df = (
    deduplicated_transactions_df.alias("transaction")
    .join(
        car_lookup_df.alias("lookup"),
        (
            col("transaction.SourceBatch")
            == col("lookup.SourceBatch")
        )
        &
        (
            col("transaction.PlateNo")
            == col("lookup.PlateNo")
        ),
        "left"
    )
    .filter(
        col("lookup.CarKey").isNull()
    )
    .select(
        col("transaction.SourceBatch"),
        col("transaction.PlateNo"),
        col("transaction.TransactionDate"),
        col("transaction.CustomerNo"),
        col("transaction.EmployeeNo"),
        col("transaction.LocationCode"),
        col("transaction.TransactionType"),
        col("transaction.Amount")
    )
    .distinct()
)

display(missing_car_references_df)

In [0]:
missing_location_references_df = (
    deduplicated_transactions_df.alias("transaction")
    .join(
        location_lookup_df.alias("lookup"),
        (
            col("transaction.SourceBatch")
            == col("lookup.SourceBatch")
        )
        &
        (
            col("transaction.LocationCode")
            == col("lookup.LocationCode")
        ),
        "left"
    )
    .filter(
        col("lookup.LocationKey").isNull()
    )
    .select(
        col("transaction.SourceBatch"),
        col("transaction.LocationCode"),
        col("transaction.TransactionDate"),
        col("transaction.PlateNo"),
        col("transaction.CustomerNo"),
        col("transaction.EmployeeNo"),
        col("transaction.TransactionType"),
        col("transaction.Amount")
    )
    .distinct()
)

display(missing_location_references_df)

In [0]:
display(
    gold_dim_car_all_df
    .join(
        missing_car_references_df
        .select("PlateNo")
        .distinct(),
        "PlateNo",
        "inner"
    )
    .orderBy(
        "PlateNo",
        "EffectiveDate"
    )
)


In [0]:
display(
    gold_dim_location_all_df
    .join(
        missing_location_references_df
        .select("LocationCode")
        .distinct(),
        "LocationCode",
        "inner"
    )
    .orderBy(
        "LocationCode",
        "EffectiveDate"
    )
)

In [0]:
display(
    deduplicated_transactions_df
    .filter(
        (col("PlateNo") == "TYT-229")
        &
        (col("CustomerNo") == 5)
        &
        (col("EmployeeNo") == 3)
    )
    .select(
        "SourceBatch",
        "PlateNo",
        "LocationCode",
        "CustomerNo",
        "EmployeeNo",
        "TransactionDate",
        "TransactionType",
        "Amount",
        "Days"
    )
    .orderBy("TransactionDate")
)

In [0]:
from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col,
    last,
    first,
    coalesce
)

rental_context_window = (
    Window
    .partitionBy(
        "PlateNo",
        "CustomerNo",
        "EmployeeNo"
    )
    .orderBy("TransactionDate")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

reverse_rental_context_window = (
    Window
    .partitionBy(
        "PlateNo",
        "CustomerNo",
        "EmployeeNo"
    )
    .orderBy("TransactionDate")
    .rowsBetween(
        Window.currentRow,
        Window.unboundedFollowing
    )
)

transactions_location_fixed_df = (
    all_transactions_hashed_df
    .withColumn(
        "PreviousKnownLocation",
        last(
            "LocationCode",
            ignorenulls=True
        ).over(rental_context_window)
    )
    .withColumn(
        "NextKnownLocation",
        first(
            "LocationCode",
            ignorenulls=True
        ).over(reverse_rental_context_window)
    )
    .withColumn(
        "LocationCode",
        coalesce(
            col("LocationCode"),
            col("PreviousKnownLocation"),
            col("NextKnownLocation")
        )
    )
    .drop(
        "PreviousKnownLocation",
        "NextKnownLocation"
    )
)

In [0]:
display(
    transactions_location_fixed_df
    .filter(
        (col("PlateNo") == "TYT-229")
        &
        (col("CustomerNo") == 5)
        &
        (col("EmployeeNo") == 3)
    )
    .select(
        "SourceBatch",
        "PlateNo",
        "LocationCode",
        "CustomerNo",
        "EmployeeNo",
        "TransactionDate",
        "TransactionType",
        "Amount",
        "Days"
    )
    .orderBy("TransactionDate")
)


In [0]:
from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col,
    last,
    first,
    coalesce
)

car_location_window = (
    Window
    .partitionBy("PlateNo")
    .orderBy("TransactionDate")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

reverse_car_location_window = (
    Window
    .partitionBy("PlateNo")
    .orderBy("TransactionDate")
    .rowsBetween(
        Window.currentRow,
        Window.unboundedFollowing
    )
)

transactions_location_fixed_df = (
    all_transactions_hashed_df
    .withColumn(
        "PreviousKnownLocation",
        last(
            "LocationCode",
            ignorenulls=True
        ).over(car_location_window)
    )
    .withColumn(
        "NextKnownLocation",
        first(
            "LocationCode",
            ignorenulls=True
        ).over(reverse_car_location_window)
    )
    .withColumn(
        "LocationCode",
        coalesce(
            col("LocationCode"),
            col("PreviousKnownLocation"),
            col("NextKnownLocation")
        )
    )
    .drop(
        "PreviousKnownLocation",
        "NextKnownLocation"
    )
)

In [0]:
display(
    transactions_location_fixed_df
    .filter(
        (col("PlateNo") == "TYT-229")
        &
        (col("CustomerNo") == 5)
        &
        (col("EmployeeNo") == 3)
    )
    .select(
        "SourceBatch",
        "PlateNo",
        "LocationCode",
        "CustomerNo",
        "EmployeeNo",
        "TransactionDate",
        "TransactionType",
        "Amount",
        "Days"
    )
    .orderBy("TransactionDate")
)


In [0]:
null_location_count = (
    transactions_location_fixed_df
    .filter(col("LocationCode").isNull())
    .count()
)

print(
    "Transactions with NULL LocationCode:",
    null_location_count
)

In [0]:
from pyspark.sql.functions import (
    sha2,
    concat_ws,
    coalesce,
    lit
)

transaction_hash_columns = [
    "PlateNo",
    "LocationCode",
    "CustomerNo",
    "EmployeeNo",
    "TransactionDate",
    "Quantity",
    "Amount",
    "TransactionType",
    "Days"
]

transactions_rehashed_df = (
    transactions_location_fixed_df
    .drop("TransactionHash")
    .withColumn(
        "TransactionHash",
        sha2(
            concat_ws(
                "||",
                *[
                    coalesce(
                        col(column_name).cast("string"),
                        lit("")
                    )
                    for column_name in transaction_hash_columns
                ]
            ),
            256
        )
    )
)

deduplicated_transactions_df = (
    transactions_rehashed_df
    .dropDuplicates(["TransactionHash"])
)

print(
    "Rows after location correction and deduplication:",
    deduplicated_transactions_df.count()
)

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

fact_rental_df = (
    resolved_fact_df
    .withColumn(
        "RentalKey",
        row_number().over(
            Window.orderBy(
                "TransactionDate",
                "TransactionHash"
            )
        )
    )
    .select(
        "RentalKey",
        "TransactionHash",
        "CarKey",
        "CustomerKey",
        "EmployeeKey",
        "LocationKey",
        "DateKey",
        "TransactionDate",
        "Quantity",
        "Amount",
        "TransactionType",
        "Days",
        "SourceBatch"
    )
)

display(
    fact_rental_df.orderBy("RentalKey")
)

In [0]:
print(
    "Final fact_rental rows:",
    fact_rental_df.count()
)

print(
    "Duplicate TransactionHash values:",
    fact_rental_df
        .groupBy("TransactionHash")
        .count()
        .filter(col("count") > 1)
        .count()
)

print(
    "NULL DateKey values:",
    fact_rental_df
        .filter(col("DateKey").isNull())
        .count()
)

fact_rental_df.groupBy(
    "SourceBatch"
).count().orderBy(
    "SourceBatch"
).show()

fact_rental_df.groupBy(
    "TransactionType"
).count().show()

In [0]:
fact_rental_path = f"{gold_base}/fact_rental"

(
    fact_rental_df.write
    .format("delta")
    .mode("overwrite")
    .save(fact_rental_path)
)

print(
    "Gold fact_rental successfully created."
)

In [0]:
gold_fact_rental_df = (
    spark.read
    .format("delta")
    .load(fact_rental_path)
)

print(
    "Saved Gold fact_rental rows:",
    gold_fact_rental_df.count()
)

display(
    gold_fact_rental_df.orderBy(
        "TransactionDate",
        "RentalKey"
    )
)

In [0]:
gold_tables = {
    "dim_car": dim_car_path,
    "dim_customer": dim_customer_path,
    "dim_employee": dim_employee_path,
    "dim_location": dim_location_path,
    "fact_rental": fact_rental_path
}

for table_name, table_path in gold_tables.items():

    table_df = (
        spark.read
        .format("delta")
        .load(table_path)
    )

    print(
        table_name,
        "rows:",
        table_df.count()
    )


In [0]:
fact_validation_df = (
    spark.read
    .format("delta")
    .load(fact_rental_path)
)

print("=" * 65)
print("AUTORENT FACT RENTAL REFERENTIAL INTEGRITY")
print("=" * 65)

print(
    "Missing CarKey values      :",
    fact_validation_df
    .filter(col("CarKey").isNull())
    .count()
)

print(
    "Missing CustomerKey values :",
    fact_validation_df
    .filter(col("CustomerKey").isNull())
    .count()
)

print(
    "Missing EmployeeKey values :",
    fact_validation_df
    .filter(col("EmployeeKey").isNull())
    .count()
)

print(
    "Missing LocationKey values :",
    fact_validation_df
    .filter(col("LocationKey").isNull())
    .count()
)

print(
    "Missing DateKey values     :",
    fact_validation_df
    .filter(col("DateKey").isNull())
    .count()
)

print("=" * 65)
print("Expected result: 0 missing keys")
print("=" * 65)

In [0]:
gold_base = "abfss://autorent@stautorentjs2026.dfs.core.windows.net/gold"
fact_rental_path = f"{gold_base}/fact_rental"

fact_check_df = (
    spark.read
    .format("delta")
    .load(fact_rental_path)
)

display(
    fact_check_df
    .filter(
        col("CarKey").isNull()
        |
        col("LocationKey").isNull()
    )
    .orderBy("TransactionDate")
)

In [0]:
# ============================================================
# AUTORENT - REBUILD AND REPAIR GOLD FACT_RENTAL
# SELF-CONTAINED REPAIR CELL
#
# This cell:
# - reloads persisted Silver and Gold Delta data
# - repairs missing transaction locations
# - recreates transaction hashes
# - resolves exact historical keys where possible
# - uses current Gold keys only as a fallback
# - validates all keys
# - overwrites fact_rental only when validation passes
# ============================================================

from functools import reduce
from pyspark.sql.functions import (
    col,
    lit,
    coalesce,
    sha2,
    concat,
    last,
    first,
    date_format,
    row_number
)
from pyspark.sql.window import Window


# ============================================================
# 1. STORAGE PATHS
# ============================================================

silver_base = (
    "abfss://autorent@stautorentjs2026"
    ".dfs.core.windows.net/silver"
)

gold_base = (
    "abfss://autorent@stautorentjs2026"
    ".dfs.core.windows.net/gold"
)

dim_car_path = f"{gold_base}/dim_car"
dim_customer_path = f"{gold_base}/dim_customer"
dim_employee_path = f"{gold_base}/dim_employee"
dim_location_path = f"{gold_base}/dim_location"
fact_rental_path = f"{gold_base}/fact_rental"


# ============================================================
# 2. HELPER FUNCTION FOR ATTRIBUTE HASHING
# ============================================================

def add_attribute_hash(dataframe, tracked_columns):

    expressions = [
        coalesce(
            col(column_name).cast("string"),
            lit("")
        )
        for column_name in tracked_columns
    ]

    combined_expression = reduce(
        lambda left_value, right_value:
            concat(
                left_value,
                lit("||"),
                right_value
            ),
        expressions
    )

    return dataframe.withColumn(
        "AttributeHash",
        sha2(
            combined_expression,
            256
        )
    )


# ============================================================
# 3. LOAD DAY 1, DAY 2, AND DAY 3 TRANSACTIONS
# ============================================================

day1_transactions = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/transactions")
    .withColumn(
        "SourceBatch",
        lit("Day1")
    )
)

day2_transactions = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/transactions")
    .withColumn(
        "SourceBatch",
        lit("Day2")
    )
)

day3_transactions = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day3/transactions")
    .withColumn(
        "SourceBatch",
        lit("Day3")
    )
)

all_transactions_df = (
    day1_transactions
    .unionByName(day2_transactions)
    .unionByName(day3_transactions)
)

print(
    "Combined transaction rows:",
    all_transactions_df.count()
)


# ============================================================
# 4. REPAIR NULL LOCATION CODES
#
# A missing LocationCode uses:
# - the previous known location for the same vehicle
# - otherwise the next known location for the same vehicle
# ============================================================

previous_location_window = (
    Window
    .partitionBy("PlateNo")
    .orderBy("TransactionDate")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

next_location_window = (
    Window
    .partitionBy("PlateNo")
    .orderBy("TransactionDate")
    .rowsBetween(
        Window.currentRow,
        Window.unboundedFollowing
    )
)

transactions_location_fixed_df = (
    all_transactions_df
    .withColumn(
        "PreviousKnownLocation",
        last(
            "LocationCode",
            ignorenulls=True
        ).over(previous_location_window)
    )
    .withColumn(
        "NextKnownLocation",
        first(
            "LocationCode",
            ignorenulls=True
        ).over(next_location_window)
    )
    .withColumn(
        "LocationCode",
        coalesce(
            col("LocationCode"),
            col("PreviousKnownLocation"),
            col("NextKnownLocation")
        )
    )
    .drop(
        "PreviousKnownLocation",
        "NextKnownLocation"
    )
)

null_location_count = (
    transactions_location_fixed_df
    .filter(
        col("LocationCode").isNull()
    )
    .count()
)

print(
    "Transactions with NULL LocationCode after repair:",
    null_location_count
)


# ============================================================
# 5. RECREATE TRANSACTION HASH
# ============================================================

transaction_hash_columns = [
    "PlateNo",
    "LocationCode",
    "CustomerNo",
    "EmployeeNo",
    "TransactionDate",
    "Quantity",
    "Amount",
    "TransactionType",
    "Days"
]

transaction_hash_expressions = [
    coalesce(
        col(column_name).cast("string"),
        lit("")
    )
    for column_name in transaction_hash_columns
]

combined_transaction_expression = reduce(
    lambda left_value, right_value:
        concat(
            left_value,
            lit("||"),
            right_value
        ),
    transaction_hash_expressions
)

deduplicated_transactions_df = (
    transactions_location_fixed_df
    .withColumn(
        "TransactionHash",
        sha2(
            combined_transaction_expression,
            256
        )
    )
    .dropDuplicates(
        ["TransactionHash"]
    )
)

print(
    "Transactions after deduplication:",
    deduplicated_transactions_df.count()
)


# ============================================================
# 6. LOAD SILVER DIMENSION SNAPSHOTS
# ============================================================

day1_cars = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/cars")
)

day2_cars = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/cars")
)

day3_cars = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day3/cars")
)


day1_customers = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/customers")
)

day2_customers = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/customers")
)

day3_customers = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day3/customers")
)


day1_employees = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/employees")
)

day2_employees = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/employees")
)

day3_employees = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day3/employees")
)


day1_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/locations")
)

day2_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day2/locations")
)

day3_locations = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/day3/locations")
)


# ============================================================
# 7. TRACKED DIMENSION ATTRIBUTES
# ============================================================

car_attributes = [
    "CarName",
    "Manufacturer",
    "YearModel",
    "CarBodyType",
    "Transmission",
    "DailyRate"
]

customer_attributes = [
    "CustomerName",
    "Address",
    "DriversLicenseNo",
    "Age",
    "Email",
    "MobileNo"
]

employee_attributes = [
    "EmployeeName",
    "Address",
    "BirthDate",
    "Email",
    "Phone"
]

location_attributes = [
    "Address1",
    "Address2",
    "City",
    "Province"
]


# ============================================================
# 8. CREATE BATCH-SPECIFIC SNAPSHOTS
# ============================================================

day1_cars_hashed = (
    add_attribute_hash(
        day1_cars,
        car_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day1")
    )
)

day2_cars_hashed = (
    add_attribute_hash(
        day2_cars,
        car_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day2")
    )
)

day3_cars_hashed = (
    add_attribute_hash(
        day3_cars,
        car_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day3")
    )
)

all_car_snapshots_df = (
    day1_cars_hashed
    .unionByName(day2_cars_hashed)
    .unionByName(day3_cars_hashed)
)


day1_customers_hashed = (
    add_attribute_hash(
        day1_customers,
        customer_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day1")
    )
)

day2_customers_hashed = (
    add_attribute_hash(
        day2_customers,
        customer_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day2")
    )
)

day3_customers_hashed = (
    add_attribute_hash(
        day3_customers,
        customer_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day3")
    )
)

all_customer_snapshots_df = (
    day1_customers_hashed
    .unionByName(day2_customers_hashed)
    .unionByName(day3_customers_hashed)
)


day1_employees_hashed = (
    add_attribute_hash(
        day1_employees,
        employee_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day1")
    )
)

day2_employees_hashed = (
    add_attribute_hash(
        day2_employees,
        employee_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day2")
    )
)

day3_employees_hashed = (
    add_attribute_hash(
        day3_employees,
        employee_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day3")
    )
)

all_employee_snapshots_df = (
    day1_employees_hashed
    .unionByName(day2_employees_hashed)
    .unionByName(day3_employees_hashed)
)


day1_locations_hashed = (
    add_attribute_hash(
        day1_locations,
        location_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day1")
    )
)

day2_locations_hashed = (
    add_attribute_hash(
        day2_locations,
        location_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day2")
    )
)

day3_locations_hashed = (
    add_attribute_hash(
        day3_locations,
        location_attributes
    )
    .withColumn(
        "SourceBatch",
        lit("Day3")
    )
)

all_location_snapshots_df = (
    day1_locations_hashed
    .unionByName(day2_locations_hashed)
    .unionByName(day3_locations_hashed)
)


# ============================================================
# 9. LOAD GOLD DIMENSIONS
# ============================================================

gold_dim_car_df = (
    spark.read
    .format("delta")
    .load(dim_car_path)
)

gold_dim_customer_df = (
    spark.read
    .format("delta")
    .load(dim_customer_path)
)

gold_dim_employee_df = (
    spark.read
    .format("delta")
    .load(dim_employee_path)
)

gold_dim_location_df = (
    spark.read
    .format("delta")
    .load(dim_location_path)
)


# ============================================================
# 10. BUILD EXACT BATCH AND HASH LOOKUPS
# ============================================================

car_lookup_df = (
    all_car_snapshots_df.alias("source")
    .join(
        gold_dim_car_df.alias("gold"),
        (
            col("source.PlateNo")
            == col("gold.PlateNo")
        )
        &
        (
            col("source.AttributeHash")
            == col("gold.AttributeHash")
        ),
        "left"
    )
    .select(
        col("source.SourceBatch"),
        col("source.PlateNo"),
        col("gold.CarKey")
    )
    .dropDuplicates(
        ["SourceBatch", "PlateNo"]
    )
)


customer_lookup_df = (
    all_customer_snapshots_df.alias("source")
    .join(
        gold_dim_customer_df.alias("gold"),
        (
            col("source.CustomerNo")
            == col("gold.CustomerNo")
        )
        &
        (
            col("source.AttributeHash")
            == col("gold.AttributeHash")
        ),
        "left"
    )
    .select(
        col("source.SourceBatch"),
        col("source.CustomerNo"),
        col("gold.CustomerKey")
    )
    .dropDuplicates(
        ["SourceBatch", "CustomerNo"]
    )
)


employee_lookup_df = (
    all_employee_snapshots_df.alias("source")
    .join(
        gold_dim_employee_df.alias("gold"),
        (
            col("source.EmployeeNo")
            == col("gold.EmployeeNo")
        )
        &
        (
            col("source.AttributeHash")
            == col("gold.AttributeHash")
        ),
        "left"
    )
    .select(
        col("source.SourceBatch"),
        col("source.EmployeeNo"),
        col("gold.EmployeeKey")
    )
    .dropDuplicates(
        ["SourceBatch", "EmployeeNo"]
    )
)


location_lookup_df = (
    all_location_snapshots_df.alias("source")
    .join(
        gold_dim_location_df.alias("gold"),
        (
            col("source.LocationCode")
            == col("gold.LocationCode")
        )
        &
        (
            col("source.AttributeHash")
            == col("gold.AttributeHash")
        ),
        "left"
    )
    .select(
        col("source.SourceBatch"),
        col("source.LocationCode"),
        col("gold.LocationKey")
    )
    .dropDuplicates(
        ["SourceBatch", "LocationCode"]
    )
)


# ============================================================
# 11. CREATE CURRENT GOLD FALLBACKS
# ============================================================

current_car_fallback_df = (
    gold_dim_car_df
    .filter(
        col("IsCurrent") == True
    )
    .select(
        "PlateNo",
        col("CarKey").alias(
            "FallbackCarKey"
        )
    )
    .dropDuplicates(
        ["PlateNo"]
    )
)

current_customer_fallback_df = (
    gold_dim_customer_df
    .filter(
        col("IsCurrent") == True
    )
    .select(
        "CustomerNo",
        col("CustomerKey").alias(
            "FallbackCustomerKey"
        )
    )
    .dropDuplicates(
        ["CustomerNo"]
    )
)

current_employee_fallback_df = (
    gold_dim_employee_df
    .filter(
        col("IsCurrent") == True
    )
    .select(
        "EmployeeNo",
        col("EmployeeKey").alias(
            "FallbackEmployeeKey"
        )
    )
    .dropDuplicates(
        ["EmployeeNo"]
    )
)

current_location_fallback_df = (
    gold_dim_location_df
    .filter(
        col("IsCurrent") == True
    )
    .select(
        "LocationCode",
        col("LocationKey").alias(
            "FallbackLocationKey"
        )
    )
    .dropDuplicates(
        ["LocationCode"]
    )
)


# ============================================================
# 12. RESOLVE FACT DIMENSION KEYS
# ============================================================

resolved_fact_df = (
    deduplicated_transactions_df.alias("fact")

    .join(
        car_lookup_df.alias("car"),
        (
            col("fact.SourceBatch")
            == col("car.SourceBatch")
        )
        &
        (
            col("fact.PlateNo")
            == col("car.PlateNo")
        ),
        "left"
    )

    .join(
        customer_lookup_df.alias("customer"),
        (
            col("fact.SourceBatch")
            == col("customer.SourceBatch")
        )
        &
        (
            col("fact.CustomerNo")
            == col("customer.CustomerNo")
        ),
        "left"
    )

    .join(
        employee_lookup_df.alias("employee"),
        (
            col("fact.SourceBatch")
            == col("employee.SourceBatch")
        )
        &
        (
            col("fact.EmployeeNo")
            == col("employee.EmployeeNo")
        ),
        "left"
    )

    .join(
        location_lookup_df.alias("location"),
        (
            col("fact.SourceBatch")
            == col("location.SourceBatch")
        )
        &
        (
            col("fact.LocationCode")
            == col("location.LocationCode")
        ),
        "left"
    )

    .join(
        current_car_fallback_df.alias(
            "car_fallback"
        ),
        col("fact.PlateNo")
        == col("car_fallback.PlateNo"),
        "left"
    )

    .join(
        current_customer_fallback_df.alias(
            "customer_fallback"
        ),
        col("fact.CustomerNo")
        == col("customer_fallback.CustomerNo"),
        "left"
    )

    .join(
        current_employee_fallback_df.alias(
            "employee_fallback"
        ),
        col("fact.EmployeeNo")
        == col("employee_fallback.EmployeeNo"),
        "left"
    )

    .join(
        current_location_fallback_df.alias(
            "location_fallback"
        ),
        col("fact.LocationCode")
        == col("location_fallback.LocationCode"),
        "left"
    )

    .select(
        col("fact.TransactionHash"),

        coalesce(
            col("car.CarKey"),
            col("car_fallback.FallbackCarKey")
        ).alias("CarKey"),

        coalesce(
            col("customer.CustomerKey"),
            col(
                "customer_fallback.FallbackCustomerKey"
            )
        ).alias("CustomerKey"),

        coalesce(
            col("employee.EmployeeKey"),
            col(
                "employee_fallback.FallbackEmployeeKey"
            )
        ).alias("EmployeeKey"),

        coalesce(
            col("location.LocationKey"),
            col(
                "location_fallback.FallbackLocationKey"
            )
        ).alias("LocationKey"),

        date_format(
            col("fact.TransactionDate"),
            "yyyyMMdd"
        )
        .cast("int")
        .alias("DateKey"),

        col("fact.TransactionDate"),
        col("fact.Quantity"),
        col("fact.Amount"),
        col("fact.TransactionType"),
        col("fact.Days"),
        col("fact.SourceBatch")
    )
)


# ============================================================
# 13. PRE-WRITE VALIDATION
# ============================================================

missing_car_count = (
    resolved_fact_df
    .filter(
        col("CarKey").isNull()
    )
    .count()
)

missing_customer_count = (
    resolved_fact_df
    .filter(
        col("CustomerKey").isNull()
    )
    .count()
)

missing_employee_count = (
    resolved_fact_df
    .filter(
        col("EmployeeKey").isNull()
    )
    .count()
)

missing_location_count = (
    resolved_fact_df
    .filter(
        col("LocationKey").isNull()
    )
    .count()
)

missing_date_count = (
    resolved_fact_df
    .filter(
        col("DateKey").isNull()
    )
    .count()
)

separator = "=".ljust(65, "=")

print(separator)
print(
    "CORRECTED FACT RENTAL - PRE-WRITE VALIDATION"
)
print(separator)

print(
    "Missing CarKey values      :",
    missing_car_count
)

print(
    "Missing CustomerKey values :",
    missing_customer_count
)

print(
    "Missing EmployeeKey values :",
    missing_employee_count
)

print(
    "Missing LocationKey values :",
    missing_location_count
)

print(
    "Missing DateKey values     :",
    missing_date_count
)

print(separator)


# ============================================================
# 14. STOP WHEN ANY KEY REMAINS UNRESOLVED
# ============================================================

all_keys_valid = (
    missing_car_count == 0
    and missing_customer_count == 0
    and missing_employee_count == 0
    and missing_location_count == 0
    and missing_date_count == 0
)

if not all_keys_valid:

    display(
        resolved_fact_df
        .filter(
            col("CarKey").isNull()
            |
            col("CustomerKey").isNull()
            |
            col("EmployeeKey").isNull()
            |
            col("LocationKey").isNull()
            |
            col("DateKey").isNull()
        )
    )

    raise ValueError(
        "Fact repair stopped because unresolved "
        "dimension keys still exist."
    )


# ============================================================
# 15. CREATE AND WRITE CORRECTED FACT_RENTAL
# ============================================================

fact_rental_df = (
    resolved_fact_df
    .withColumn(
        "RentalKey",
        row_number().over(
            Window.orderBy(
                "TransactionDate",
                "TransactionHash"
            )
        )
    )
    .select(
        "RentalKey",
        "TransactionHash",
        "CarKey",
        "CustomerKey",
        "EmployeeKey",
        "LocationKey",
        "DateKey",
        "TransactionDate",
        "Quantity",
        "Amount",
        "TransactionType",
        "Days",
        "SourceBatch"
    )
)

(
    fact_rental_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(fact_rental_path)
)

print(
    "Corrected Gold fact_rental "
    "successfully written."
)


# ============================================================
# 16. VALIDATE THE SAVED GOLD FACT
# ============================================================

fact_validation_df = (
    spark.read
    .format("delta")
    .load(fact_rental_path)
)

print(separator)
print(
    "AUTORENT FACT RENTAL REFERENTIAL INTEGRITY"
)
print(separator)

print(
    "Missing CarKey values      :",
    fact_validation_df
    .filter(
        col("CarKey").isNull()
    )
    .count()
)

print(
    "Missing CustomerKey values :",
    fact_validation_df
    .filter(
        col("CustomerKey").isNull()
    )
    .count()
)

print(
    "Missing EmployeeKey values :",
    fact_validation_df
    .filter(
        col("EmployeeKey").isNull()
    )
    .count()
)

print(
    "Missing LocationKey values :",
    fact_validation_df
    .filter(
        col("LocationKey").isNull()
    )
    .count()
)

print(
    "Missing DateKey values     :",
    fact_validation_df
    .filter(
        col("DateKey").isNull()
    )
    .count()
)

print(separator)
print("Expected result: 0 missing keys")
print(separator)

In [0]:
from pyspark.sql.functions import upper, trim

transactions_location_fixed_df = (
    transactions_location_fixed_df
    .withColumn(
        "PlateNo",
        upper(trim(col("PlateNo")))
    )
    .withColumn(
        "LocationCode",
        upper(trim(col("LocationCode")))
    )
)

In [0]:
day1_cars = (
    day1_cars
    .withColumn(
        "PlateNo",
        upper(trim(col("PlateNo")))
    )
)

day2_cars = (
    day2_cars
    .withColumn(
        "PlateNo",
        upper(trim(col("PlateNo")))
    )
)

day3_cars = (
    day3_cars
    .withColumn(
        "PlateNo",
        upper(trim(col("PlateNo")))
    )
)

day1_locations = (
    day1_locations
    .withColumn(
        "LocationCode",
        upper(trim(col("LocationCode")))
    )
)

day2_locations = (
    day2_locations
    .withColumn(
        "LocationCode",
        upper(trim(col("LocationCode")))
    )
)

day3_locations = (
    day3_locations
    .withColumn(
        "LocationCode",
        upper(trim(col("LocationCode")))
    )
)

In [0]:
# ============================================================
# CREATE INFERRED DIM_CAR MEMBER FOR TYT-999
# Schema-safe version
# ============================================================

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp
)

gold_base = (
    "abfss://autorent@stautorentjs2026"
    ".dfs.core.windows.net/gold"
)

dim_car_path = f"{gold_base}/dim_car"

gold_dim_car_df = (
    spark.read
    .format("delta")
    .load(dim_car_path)
)

missing_plate_number = "TYT-999"

target_car_key_type = (
    gold_dim_car_df
    .schema["CarKey"]
    .dataType
    .simpleString()
)

print(
    "Existing dim_car CarKey type:",
    target_car_key_type
)

existing_member_count = (
    gold_dim_car_df
    .filter(
        col("PlateNo") == missing_plate_number
    )
    .count()
)

if existing_member_count == 0:

    maximum_car_key = (
        gold_dim_car_df
        .selectExpr(
            "COALESCE(MAX(CarKey), 0) AS MaximumCarKey"
        )
        .first()["MaximumCarKey"]
    )

    inferred_car_base_df = (
        spark.createDataFrame(
            [
                (
                    "TYT-999",
                    "Unknown Car",
                    "Unknown Manufacturer",
                    None,
                    "Unknown Body Type",
                    "Unknown Transmission",
                    None,
                    "INFERRED-TYT-999"
                )
            ],
            """
            PlateNo string,
            CarName string,
            Manufacturer string,
            YearModel integer,
            CarBodyType string,
            Transmission string,
            DailyRate decimal(10,2),
            AttributeHash string
            """
        )
    )

    inferred_car_df = (
        inferred_car_base_df
        .withColumn(
            "CarKey",
            lit(maximum_car_key + 1)
            .cast(target_car_key_type)
        )
        .withColumn(
            "EffectiveDate",
            current_timestamp()
        )
        .withColumn(
            "ExpirationDate",
            lit(None).cast("timestamp")
        )
        .withColumn(
            "IsCurrent",
            lit(True)
        )
        .select(
            "CarKey",
            "PlateNo",
            "CarName",
            "Manufacturer",
            "YearModel",
            "CarBodyType",
            "Transmission",
            "DailyRate",
            "AttributeHash",
            "EffectiveDate",
            "ExpirationDate",
            "IsCurrent"
        )
    )

    print("Incoming inferred-row schema:")
    inferred_car_df.printSchema()

    (
        inferred_car_df.write
        .format("delta")
        .mode("append")
        .save(dim_car_path)
    )

    print(
        "SUCCESS: Inferred dim_car member created for TYT-999"
    )

else:

    print(
        "SKIPPED: TYT-999 already exists in dim_car"
    )

In [0]:
gold_dim_car_df = (
    spark.read
    .format("delta")
    .load(dim_car_path)
)

display(
    gold_dim_car_df
    .filter(
        col("PlateNo") == "TYT-999"
    )
    .select(
        "CarKey",
        "PlateNo",
        "CarName",
        "Manufacturer",
        "YearModel",
        "CarBodyType",
        "Transmission",
        "DailyRate",
        "EffectiveDate",
        "ExpirationDate",
        "IsCurrent"
    )
)

In [0]:
# ============================================================
# AUTORENT FINAL ETL VALIDATION
# Reads the latest persisted Gold tables
# ============================================================

from pyspark.sql.functions import col

gold_base = (
    "abfss://autorent@stautorentjs2026"
    ".dfs.core.windows.net/gold"
)

dim_car_path = f"{gold_base}/dim_car"
dim_customer_path = f"{gold_base}/dim_customer"
dim_employee_path = f"{gold_base}/dim_employee"
dim_location_path = f"{gold_base}/dim_location"
fact_rental_path = f"{gold_base}/fact_rental"


# ------------------------------------------------------------
# Reload latest Gold dimensions
# ------------------------------------------------------------

dimension_checks = [
    (
        "dim_car",
        dim_car_path,
        "PlateNo"
    ),
    (
        "dim_customer",
        dim_customer_path,
        "CustomerNo"
    ),
    (
        "dim_employee",
        dim_employee_path,
        "EmployeeNo"
    ),
    (
        "dim_location",
        dim_location_path,
        "LocationCode"
    )
]

validation_results = []


# ------------------------------------------------------------
# Validate SCD dimensions
# ------------------------------------------------------------

for dimension_name, dimension_path, business_key in dimension_checks:

    dimension_df = (
        spark.read
        .format("delta")
        .load(dimension_path)
    )

    duplicate_current_count = (
        dimension_df
        .filter(
            col("IsCurrent") == True
        )
        .groupBy(
            business_key
        )
        .count()
        .filter(
            col("count") > 1
        )
        .count()
    )

    null_business_key_count = (
        dimension_df
        .filter(
            col(business_key).isNull()
        )
        .count()
    )

    dimension_passed = (
        duplicate_current_count == 0
        and
        null_business_key_count == 0
    )

    validation_results.append(
        (
            dimension_name,
            dimension_passed
        )
    )


# ------------------------------------------------------------
# Reload latest persisted fact_rental
# ------------------------------------------------------------

fact_validation_df = (
    spark.read
    .format("delta")
    .load(fact_rental_path)
)


# ------------------------------------------------------------
# Validate all Fact dimension keys
# ------------------------------------------------------------

missing_car_count = (
    fact_validation_df
    .filter(
        col("CarKey").isNull()
    )
    .count()
)

missing_customer_count = (
    fact_validation_df
    .filter(
        col("CustomerKey").isNull()
    )
    .count()
)

missing_employee_count = (
    fact_validation_df
    .filter(
        col("EmployeeKey").isNull()
    )
    .count()
)

missing_location_count = (
    fact_validation_df
    .filter(
        col("LocationKey").isNull()
    )
    .count()
)

missing_date_count = (
    fact_validation_df
    .filter(
        col("DateKey").isNull()
    )
    .count()
)

fact_passed = (
    missing_car_count == 0
    and
    missing_customer_count == 0
    and
    missing_employee_count == 0
    and
    missing_location_count == 0
    and
    missing_date_count == 0
)

validation_results.append(
    (
        "fact_rental",
        fact_passed
    )
)


# ------------------------------------------------------------
# Print final result
# ------------------------------------------------------------

separator = "=".ljust(
    60,
    "="
)

print(separator)
print("AUTORENT FINAL ETL VALIDATION")
print(separator)

for dataset_name, passed in validation_results:

    if passed:
        status = "PASSED"
    else:
        status = "FAILED"

    print(
        f"{dataset_name:<20} {status}"
    )

print(separator)

overall_passed = all(
    result[1]
    for result in validation_results
)

if overall_passed:

    print(
        "OVERALL RESULT: PASSED"
    )

else:

    print(
        "OVERALL RESULT: REVIEW REQUIRED"
    )

print(separator)